# c3-gen-union (K1): decoder per predicted formula + mutator edits, on one leak-free scale

Campaign `20261015-c3-gen-slot2`, gate K1 (`research/harness/gate_report.md`, block 2026-10-15 + amendment).

Per pseudo_c3 molecule this notebook reads `p4_inputs.pkl` (written locally by `build/novel_p4_inputs.py` from the
pool-banned pseudo_c3 dump) and produces:

| leg | what | score |
|---|---|---|
| `dec` | `search_masked_beam` per **predicted** formula (no oracle), beam `BEAM` | the beam's own masked log p |
| `dec256` | the same at beam 256 on a `SUB256` subset, for the beam-size cost of K4 | idem |
| `mut` | c3v2 analog edits from the banned Engine 2 analogs | `teacher_logp` (equal to the beam score on a leaf) |

**No FPNet anywhere** (leaky on pseudo_c3). Raw lists go to Drive `casmi_2026/_novel/p4/raw/`; ranking with the
metric's tautomer-canonical IK14 and the K1 verdict run locally (`build/novel_p4_score.py`). Resumable.

In [ ]:
import os, sys, json, time, hashlib, pickle, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

if not os.path.isdir("/content/drive/MyDrive"):     # colab_run.py mounts before exec; a notebook mounts here
    from google.colab import drive
    drive.mount("/content/drive")

T_START = time.time()
TIME_BUDGET_S = float(os.environ.get("CASMI_TIME_BUDGET_S", "inf"))  # stop generating, flush, exit before the cutoff
ROOT = Path("/content/drive/MyDrive/casmi_2026")
P0 = ROOT / "_novel" / "p0"
P1 = ROOT / "_novel" / "p1"
P3 = ROOT / "_novel" / "p3"
EMB = ROOT / "_novel" / "emb_dbec3a0_lx"
OUT = ROOT / "_novel" / "p4"
RAW = OUT / "raw"
RDKIT_WHL = next(iter(sorted((ROOT / "casmi26-offline-rdkit-2026033").glob("rdkit-*.whl"))), None)
SRC = Path("/content/novel_p4/src")
for p in (RAW, SRC):
    p.mkdir(parents=True, exist_ok=True)

SELFIES_PIN, RDKIT_PIN = "2.1.2", "2026.03.3"
N_CORPUS_ROWS = 1_323_878
BEAM = 64                 # production-feasible beam (K4); dec256 below measures what beam 256 would add
SUB256 = 150              # molecules re-run at beam 256 (first SUB256 by molecule_id)
CHUNK = 25                # flush to Drive every CHUNK molecules
assert (OUT / "p4_inputs.pkl").exists(), "upload p4_inputs.pkl (build/novel_p4_inputs.py) to casmi_2026/_novel/p4/"
for p in (P3 / "decoder.pt", P0 / "selfies_vocab.json", P1 / "pseudo_c3.parquet", EMB):
    assert p.exists(), p


def sh(cmd, check=True):
    print("$", cmd[:300], flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode and check:
        print(r.stderr[-3000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout

## Kernel packages and GPU (pins as in P3)

In [ ]:
def kernel_version(mod):
    r = subprocess.run([sys.executable, "-c", f"import {mod}; print({mod}.__version__)"], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None


def v_match(v, target):
    return v is not None and (v == target or v.replace(".0", ".") == target.replace(".0", "."))


if not v_match(kernel_version("rdkit"), RDKIT_PIN):
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if RDKIT_WHL is not None and py_tag in RDKIT_WHL.name:
        sh(f"{sys.executable} -m pip install -q --no-deps {RDKIT_WHL}", check=False)
    if not v_match(kernel_version("rdkit"), RDKIT_PIN):
        sh(f"{sys.executable} -m pip install -q rdkit==2026.3.3")
sh(f"{sys.executable} -m pip install -q selfies=={SELFIES_PIN}", check=False)

import torch
import rdkit, selfies as sf
assert v_match(rdkit.__version__, RDKIT_PIN), rdkit.__version__
assert torch.cuda.is_available(), "P4 needs a GPU runtime"
DEV = torch.device("cuda")
print(dict(torch=torch.__version__, gpu=torch.cuda.get_device_name(0), rdkit=rdkit.__version__, selfies=sf.__version__))

## Project sources (embedded verbatim from `build/` by `build/mk_novel_p4_colab.py`)

In [ ]:
NOVEL_DECODER_SRC = '"""specs_novel single source of truth for the SELFIES side and the decoder (plan §4, §5.5).\n\n    SelfiesVocab        token <-> id, the same ids as MassSpecGym\'s SelfiesTokenizer (JSON, no `tokenizers`)\n    TokenTable          per-token grammar facts, precomputed with selfies\' own grammar_rules\n    Tracker             incremental SELFIES derivation state -> legal-token mask (plan §4.1-4.2)\n    reference_state     the slow truth: selfies.decoder(prefix) + RDKit counts (P0.4 parity)\n    walk_truth          mask soundness: every true next token legal, </s> legal at the end (P0.4)\n    prep_structure      stereo/isotope strip -> canonical SMILES -> SELFIES (A9, P0.3)\n\nThe Tracker replays `selfies.decoder._derive_mol_from_symbols` token by token. That function is\nrecursive (a branch derives a sub-chain of Q+1 symbols); here its call stack is an explicit list of\nframes [state, prev_atom, budget]. Every consumed symbol lowers the budget of every open frame, the\nway each caller adds its child\'s n_derived; a frame closes when its budget reaches 0. A frame whose\nstate becomes None (no free valence left) swallows the rest of its budget, and in the main chain that\nmeans every later symbol -- SELFIES silently skips those atoms (A7), so the mask never offers them.\n\nMasks (plan §4.2):\n    pending index symbols      only the 16 INDEX_ALPHABET symbols (the encoder emits nothing else there)\n    atom token, element e      frame active, the atom would be placed, and remaining[e] > 0\n    [Branch*]                  frame state > 1 (otherwise selfies ignores it)\n    [Ring*]                    frame state > 0\n    </s>                       no pending index and every heavy-atom count exactly used\n`unsat()` is rings + pi bonds of the molecule the prefix decodes to, with the ring bonds formed the way\n`_form_rings_bilocally` forms them at the end (orders capped by the free valence left). It feeds the\noptional DBE prune, which is only applied where P0.4 shows it sound (A8).\n\nP3 part (below the P0 part, specs_novel/gate_report.md -> G3):\n    DreamsSelfiesDecoder  memory = per-spectrum tokens (+ adduct, + CE bin) plus a formula token,\n                          no positional encoding on the memory, tied in/out token embeddings\n    search_prefix_seeded  arm A: masked expansion to depth L_SEED, then sampled rollouts per prefix\n    search_masked_beam    arm B (alpha = 0) and arm C (alpha > 0, cross-beam scaffold penalty)\n    select_top25          "maxmin" (diversity, what G3 is judged on) or "logp"\n    max_tanimoto          the G3 metric, on MassSpecGym\'s fingerprint\n\nNeeds numpy + selfies (pinned) for the P0 part; RDKit for reference_state / prep_structure / ik14 /\nthe selection helpers; torch for the P3 part.\n"""\nimport json\nimport re\n\nimport numpy as np\nimport selfies as sf\nfrom selfies.constants import INDEX_CODE\nfrom selfies.grammar_rules import (get_index_from_selfies, next_atom_state, next_branch_state,\n                                   next_ring_state, process_atom_symbol, process_branch_symbol,\n                                   process_ring_symbol)\n\nPAD, SOS, EOS, UNK = "<pad>", "<s>", "</s>", "<unk>"\nALLOWED_ELEMENTS = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")\nDBE_SAFE_ELEMENTS = frozenset(("C", "H", "N", "O", "F", "Cl", "Br", "I"))   # no S / P (plan §4.2)\nK_SPECIAL, K_ATOM, K_BRANCH, K_RING, K_OTHER = 0, 1, 2, 3, 4\n_FORMULA_RE = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef check_constraints():\n    """The token table and the tracker assume selfies\' default semantic constraints."""\n    assert sf.get_semantic_constraints() == sf.get_preset_constraints("default"), \\\n        "selfies semantic constraints are not the default preset"\n\n\n# --------------------------------------------------------------------------------------------- #\n# Vocabulary\n# --------------------------------------------------------------------------------------------- #\n\ndef vocab_from_alphabet(alphabet):\n    """SelfiesTokenizer\'s id layout (massspecgym/models/tokenizers.py:79-85 + add_special_tokens):\n    sorted alphabet 0..n-1, <unk> n, then <pad> <s> </s> as the next added tokens."""\n    alphabet = sorted(set(alphabet) - {PAD, SOS, EOS, UNK})\n    tok2id = {s: i for i, s in enumerate(alphabet)}\n    tok2id[UNK] = len(tok2id)\n    for s in (PAD, SOS, EOS):\n        tok2id[s] = len(tok2id)\n    return tok2id\n\n\nclass SelfiesVocab:\n    def __init__(self, tok2id):\n        ids = sorted(tok2id.values())\n        assert ids == list(range(len(ids))), "token ids must be 0..n-1"\n        for s in (PAD, SOS, EOS, UNK):\n            assert s in tok2id, f"special token {s} missing"\n        self.tok2id = dict(tok2id)\n        self.id2tok = [None] * len(ids)\n        for s, i in self.tok2id.items():\n            self.id2tok[i] = s\n        self.pad_id, self.sos_id, self.eos_id, self.unk_id = (self.tok2id[s] for s in (PAD, SOS, EOS, UNK))\n\n    def __len__(self):\n        return len(self.id2tok)\n\n    @classmethod\n    def from_alphabet(cls, alphabet):\n        return cls(vocab_from_alphabet(alphabet))\n\n    @classmethod\n    def from_json(cls, path):\n        with open(path, encoding="utf-8") as f:\n            return cls(json.load(f))\n\n    def to_json(self, path):\n        with open(path, "w", encoding="utf-8") as f:\n            json.dump(self.tok2id, f, sort_keys=True, indent=0)\n\n    def encode(self, selfies_str):\n        """SELFIES string -> ids, no <s>/</s>."""\n        return [self.tok2id.get(t, self.unk_id) for t in sf.split_selfies(selfies_str)]\n\n    def decode(self, ids):\n        """ids -> SELFIES string; special tokens dropped."""\n        return "".join(self.id2tok[i] for i in ids if i not in (self.pad_id, self.sos_id, self.eos_id,\n                                                                self.unk_id))\n\n\n# --------------------------------------------------------------------------------------------- #\n# Token table\n# --------------------------------------------------------------------------------------------- #\n\nclass TokenTable:\n    """Per-token facts over a vocabulary, from selfies\' own grammar_rules (so they follow the pin)."""\n\n    def __init__(self, vocab):\n        check_constraints()\n        n = len(vocab)\n        self.vocab = vocab\n        self.kind = np.full(n, K_OTHER, np.int8)\n        self.elem = np.full(n, -1, np.int16)           # index into self.elements\n        self.bond = np.zeros(n, np.int8)               # requested bond order (atom / branch / ring type)\n        self.cap = np.zeros(n, np.int8)                # atom bonding capacity (after explicit H, charge)\n        self.nidx = np.zeros(n, np.int8)               # index symbols read after a branch / ring\n        self.is_index = np.zeros(n, bool)              # one of the 16 INDEX_ALPHABET symbols\n        self.index_code = np.zeros(n, np.int16)\n        elements = []\n        for i, t in enumerate(vocab.id2tok):\n            if t in (PAD, SOS, EOS, UNK):\n                self.kind[i] = K_SPECIAL\n                continue\n            if t in INDEX_CODE:\n                self.is_index[i] = True\n                self.index_code[i] = INDEX_CODE[t]\n            if "ch" == t[-4:-2]:                       # the same dispatch order as the decoder\n                out = process_branch_symbol(t)\n                if out is not None:\n                    self.kind[i], self.bond[i], self.nidx[i] = K_BRANCH, out[0], out[1]\n            elif "ng" == t[-4:-2]:\n                out = process_ring_symbol(t)\n                if out is not None:\n                    self.kind[i], self.bond[i], self.nidx[i] = K_RING, out[0], out[1]\n            elif "eps" in t:\n                pass\n            else:\n                out = process_atom_symbol(t)\n                if out is not None:\n                    (bond, _), atom = out\n                    if atom.element not in elements:\n                        elements.append(atom.element)\n                    self.kind[i] = K_ATOM\n                    self.elem[i] = elements.index(atom.element)\n                    self.bond[i] = bond\n                    self.cap[i] = atom.bonding_capacity\n        self.elements = tuple(elements)\n        self.elem_ix = {e: j for j, e in enumerate(self.elements)}\n        self.h_ix = self.elem_ix.get("H", -1)\n        self.is_atom = self.kind == K_ATOM\n        self.is_branch = self.kind == K_BRANCH\n        self.is_ring = self.kind == K_RING\n        self.atom_cap_pos = self.is_atom & (self.cap > 0)\n        self.elem_safe = np.maximum(self.elem, 0)\n\n    def counts_vector(self, counts):\n        """{element: count} -> vector over self.elements, plus the count of elements the vocab lacks."""\n        v = np.zeros(len(self.elements), np.int32)\n        missing = 0\n        for e, c in counts.items():\n            if e in self.elem_ix:\n                v[self.elem_ix[e]] = c\n            elif e != "H":\n                missing += c\n        return v, missing\n\n\n# --------------------------------------------------------------------------------------------- #\n# Tracker\n# --------------------------------------------------------------------------------------------- #\n\nINF = float("inf")\n\n\nclass Tracker:\n    """Persistent SELFIES derivation state. `push` returns a new Tracker; the old one is unchanged.\n\n    formula: {element: count} incl. H, or None (then only counting, no count masks)."""\n\n    __slots__ = ("tt", "frames", "pending", "caps", "bc", "bonds", "rings", "counts", "target",\n                 "missing", "n_tokens", "broken")\n\n    def __init__(self, table, formula=None):\n        self.tt = table\n        self.frames = [[0, None, INF]]          # [state (int | None), prev atom index, budget]\n        self.pending = None                     # [kind, n_left, symbols, payload]\n        self.caps = []                          # bonding capacity per placed atom\n        self.bc = []                            # bond order sum per atom (chain bonds only)\n        self.bonds = {}                         # (a, b) a < b -> order (chain bonds only)\n        self.rings = []                         # (left atom, right atom, order), formed at the end\n        self.counts = np.zeros(len(table.elements), np.int32)\n        if formula is None:\n            self.target, self.missing = None, 0\n        else:\n            self.target, self.missing = table.counts_vector(formula)\n        self.n_tokens = 0\n        self.broken = False\n\n    def _copy(self):\n        t = Tracker.__new__(Tracker)\n        t.tt = self.tt\n        t.frames = [f[:] for f in self.frames]\n        t.pending = None if self.pending is None else [self.pending[0], self.pending[1],\n                                                       self.pending[2][:], self.pending[3]]\n        t.caps, t.bc, t.bonds, t.rings = self.caps[:], self.bc[:], dict(self.bonds), self.rings[:]\n        t.counts = self.counts.copy()\n        t.target, t.missing, t.n_tokens, t.broken = self.target, self.missing, self.n_tokens, self.broken\n        return t\n\n    # -- derivation -------------------------------------------------------------------------- #\n\n    def _consume(self):\n        for f in self.frames:\n            f[2] -= 1\n\n    def _close_exhausted(self):\n        while len(self.frames) > 1 and self.frames[-1][2] <= 0:\n            self.frames.pop()\n\n    def _add_atom(self, cap):\n        self.caps.append(int(cap))\n        self.bc.append(0)\n        return len(self.caps) - 1\n\n    def push(self, tok_id):\n        t = self._copy()\n        t._apply(int(tok_id))\n        return t\n\n    def _apply(self, i):\n        tt = self.tt\n        self.n_tokens += 1\n        self._consume()\n        if self.pending is not None:                      # an index symbol of a branch / ring\n            p = self.pending\n            p[2].append(tt.vocab.id2tok[i])\n            p[1] -= 1\n            if p[1] == 0:\n                self.pending = None\n                q = get_index_from_selfies(*p[2])\n                f = self.frames[-1]\n                if p[0] == K_BRANCH:\n                    self.frames.append([p[3], f[1], q + 1])\n                else:\n                    lidx = max(0, f[1] - (q + 1))\n                    self.rings.append((lidx, f[1], p[3]))\n            self._close_exhausted()\n            return\n        f = self.frames[-1]\n        state = f[0]\n        kind = tt.kind[i]\n        if state is None or kind in (K_SPECIAL, K_OTHER):\n            if kind == K_OTHER and state is not None and "eps" in tt.vocab.id2tok[i]:\n                f[0] = 0 if state == 0 else None\n            elif kind == K_SPECIAL:\n                self.broken = True                        # specials never occur inside a SELFIES string\n        elif kind == K_BRANCH:\n            if state > 1:\n                binit, f[0] = next_branch_state(int(tt.bond[i]), state)\n                self.pending = [K_BRANCH, int(tt.nidx[i]), [], binit]\n        elif kind == K_RING:\n            if state != 0:\n                order, f[0] = next_ring_state(int(tt.bond[i]), state)\n                self.pending = [K_RING, int(tt.nidx[i]), [], order]\n        else:                                             # atom\n            cap = int(tt.cap[i])\n            order, nxt = next_atom_state(int(tt.bond[i]), cap, state)\n            if order == 0:\n                if state == 0:\n                    f[1] = self._add_atom(cap)\n                    self.counts[tt.elem[i]] += 1\n                else:\n                    self.broken = True                    # selfies would bond from an unplaced atom\n            else:\n                a = self._add_atom(cap)\n                self.counts[tt.elem[i]] += 1\n                src = f[1]\n                self.bonds[(src, a)] = order\n                self.bc[src] += order\n                self.bc[a] += order\n                f[1] = a\n            f[0] = nxt\n        self._close_exhausted()\n\n    # -- queries ----------------------------------------------------------------------------- #\n\n    def remaining(self):\n        return None if self.target is None else self.target - self.counts\n\n    def heavy_done(self):\n        r = self.remaining()\n        if r is None:\n            return True\n        r = r.copy()\n        if self.tt.h_ix >= 0:\n            r[self.tt.h_ix] = 0\n        return self.missing == 0 and not r.any()\n\n    def eos_legal(self):\n        return (not self.broken and self.pending is None and len(self.caps) > 0 and self.heavy_done())\n\n    def legal_mask(self):\n        """numpy bool over the vocabulary."""\n        tt = self.tt\n        n = len(tt.kind)\n        if self.broken:\n            return np.zeros(n, bool)\n        if self.pending is not None:\n            return tt.is_index.copy()\n        state = self.frames[-1][0]\n        mask = np.zeros(n, bool)\n        if state is not None:\n            placed = tt.is_atom if state == 0 else tt.atom_cap_pos\n            if self.target is not None:\n                placed = placed & (self.remaining()[tt.elem_safe] > 0)\n            mask |= placed\n            if state > 1:\n                mask |= tt.is_branch\n            if state > 0:\n                mask |= tt.is_ring\n        if self.eos_legal():\n            mask[tt.vocab.eos_id] = True\n        return mask\n\n    def unsat(self):\n        """Rings + pi bonds of the molecule this prefix decodes to (selfies\' _form_rings_bilocally)."""\n        n = len(self.caps)\n        if n == 0:\n            return 0\n        bc, bonds = self.bc[:], dict(self.bonds)\n        for l, r, order in self.rings:\n            if l == r:\n                continue\n            lfree, rfree = self.caps[l] - bc[l], self.caps[r] - bc[r]\n            if lfree <= 0 or rfree <= 0:\n                continue\n            order = min(order, lfree, rfree)\n            key = (min(l, r), max(l, r))\n            if key in bonds:\n                new = min(order + bonds[key], 3)\n                bc[l] += new - bonds[key]\n                bc[r] += new - bonds[key]\n                bonds[key] = new\n            else:\n                bonds[key] = order\n                bc[l] += order\n                bc[r] += order\n        return sum(bonds.values()) - n + 1\n\n    def element_counts(self):\n        return {e: int(c) for e, c in zip(self.tt.elements, self.counts) if c}\n\n\n# --------------------------------------------------------------------------------------------- #\n# Formulae\n# --------------------------------------------------------------------------------------------- #\n\ndef parse_formula(formula):\n    counts = {}\n    for e, c in _FORMULA_RE.findall(formula):\n        counts[e] = counts.get(e, 0) + (int(c) if c else 1)\n    return counts\n\n\ndef dbe(counts):\n    """Rings + pi bonds implied by a formula with standard valences (C4 N3 O2 X1 H1)."""\n    x = sum(counts.get(e, 0) for e in ("F", "Cl", "Br", "I"))\n    return 1 + counts.get("C", 0) - (counts.get("H", 0) + x) / 2 + counts.get("N", 0) / 2\n\n\ndef dbe_prune_applies(counts):\n    return set(e for e, c in counts.items() if c) <= DBE_SAFE_ELEMENTS\n\n\n# --------------------------------------------------------------------------------------------- #\n# RDKit side: structure prep, ik14, reference tracker\n# --------------------------------------------------------------------------------------------- #\n\ndef _chem():\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog("rdApp.*")\n    return Chem\n\n\ndef mol_formula(mol):\n    """{element: count} incl. total H (implicit + explicit) of an RDKit mol."""\n    counts = {}\n    for a in mol.GetAtoms():\n        s = a.GetSymbol()\n        counts[s] = counts.get(s, 0) + 1\n        h = a.GetTotalNumHs()\n        if h:\n            counts["H"] = counts.get("H", 0) + h\n    return counts\n\n\ndef ik14(smiles):\n    Chem = _chem()\n    m = Chem.MolFromSmiles(smiles) if smiles else None\n    if m is None:\n        return None\n    k = Chem.MolToInchiKey(m)\n    return k[:14] if k else None\n\n\ndef prep_structure(smiles, elements=ALLOWED_ELEMENTS):\n    """-> (status, canonical_smiles, selfies, formula). status \'ok\' or the reason it is excluded:\n    \'parse\', \'multi\' (more than one component), \'element\', \'encode\' (selfies constraints)."""\n    Chem = _chem()\n    m = Chem.MolFromSmiles(smiles) if smiles else None\n    if m is None:\n        return "parse", None, None, None\n    if len(Chem.GetMolFrags(m)) > 1:\n        return "multi", None, None, None\n    if any(a.GetSymbol() not in elements for a in m.GetAtoms()):\n        return "element", None, None, None\n    Chem.RemoveStereochemistry(m)\n    for a in m.GetAtoms():\n        a.SetIsotope(0)\n    can = Chem.MolToSmiles(m)\n    m = Chem.MolFromSmiles(can)\n    try:\n        s = sf.encoder(can, strict=True)\n    except Exception:\n        return "encode", can, None, None\n    if s is None:\n        return "encode", can, None, None\n    return "ok", can, s, mol_formula(m)\n\n\ndef reference_state(tokens):\n    """Slow truth for a token prefix: (element counts incl. explicit H atoms, rings + pi bonds)."""\n    Chem = _chem()\n    smi = sf.decoder("".join(tokens))\n    if not smi:\n        return {}, 0\n    m = Chem.MolFromSmiles(smi, sanitize=False)\n    counts = {}\n    for a in m.GetAtoms():\n        counts[a.GetSymbol()] = counts.get(a.GetSymbol(), 0) + 1\n    order = sum(int(b.GetBondTypeAsDouble()) for b in m.GetBonds())\n    n_comp = len(Chem.GetMolFrags(m, sanitizeFrags=False))\n    return counts, order - m.GetNumAtoms() + n_comp\n\n\ndef tracker_parity(table, tokens):\n    """Fast vs reference at every prefix. -> list of (prefix length, what, fast, reference)."""\n    vocab = table.vocab\n    t = Tracker(table)\n    bad = []\n    for k, tok in enumerate(tokens, 1):\n        t = t.push(vocab.tok2id[tok])\n        ref_counts, ref_u = reference_state(tokens[:k])\n        if t.element_counts() != ref_counts:\n            bad.append((k, "counts", t.element_counts(), ref_counts))\n        if t.pending is None and t.unsat() != ref_u:\n            bad.append((k, "unsat", t.unsat(), ref_u))\n        if t.broken:\n            bad.append((k, "broken", None, None))\n    return bad\n\n\ndef walk_truth(table, tokens, formula, max_len=None, dbe_prune=False):\n    """Mask soundness for one truth. -> list of (position, rule, token) violations.\n\n    rules: \'count\' (atom masked, element exhausted), \'grammar\' (masked for any other reason),\n    \'eos\' (</s> not legal after the last token), \'length\' (> max_len), \'dbe\' (unsat above DBE)."""\n    vocab = table.vocab\n    t = Tracker(table, formula)\n    limit = dbe(formula) if dbe_prune and dbe_prune_applies(formula) else None\n    bad = []\n    for k, tok in enumerate(tokens):\n        i = vocab.tok2id.get(tok, vocab.unk_id)\n        if not t.legal_mask()[i]:\n            r = t.remaining()\n            count_rule = (table.kind[i] == K_ATOM and t.pending is None and r is not None\n                          and r[table.elem_safe[i]] <= 0)\n            bad.append((k, "count" if count_rule else "grammar", tok))\n        t = t.push(i)\n        if limit is not None and t.pending is None and t.unsat() > limit + 1e-9:\n            bad.append((k, "dbe", tok))\n    if not t.eos_legal():\n        bad.append((len(tokens), "eos", None))\n    if max_len is not None and len(tokens) > max_len:\n        bad.append((len(tokens), "length", None))\n    return bad\n\n\n# --------------------------------------------------------------------------------------------- #\n# P3: memory, decoder, batched constrained search (torch)\n#\n# Everything below is the P3 part (specs_novel/gate_report.md -> G3). The P0 part above stays\n# torch-free; torch is imported here because every consumer of this part needs it anyway (Colab\n# training, the Kaggle worker, the local tests).\n# --------------------------------------------------------------------------------------------- #\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n# must equal simsim_common.TEST_ADDUCTS -- test_novel_p3.check_adducts asserts it\nTEST_ADDUCTS = ("[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",\n                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-")\nCE_BINS = (15.0, 23.0, 36.0, 44.0)       # P0.2 quantiles; 5 intervals + "unknown" = 6 rows\nN_CE_BINS = len(CE_BINS) + 2             # digitize gives 0..4, and 5 is "unknown"\nCE_UNKNOWN = len(CE_BINS) + 1\nFORMULA_SCALE = 10.0                     # counts are divided by this before the formula MLP\n\n\ndef ce_bin(ce, bins=CE_BINS):\n    """Collision energy in eV -> bin index. None / NaN / <= 0 -> CE_UNKNOWN."""\n    if ce is None:\n        return CE_UNKNOWN\n    try:\n        v = float(ce)\n    except (TypeError, ValueError):\n        return CE_UNKNOWN\n    if not np.isfinite(v) or v <= 0.0:\n        return CE_UNKNOWN\n    return int(np.digitize(v, bins))\n\n\ndef formula_vector(counts, elements=ALLOWED_ELEMENTS):\n    """{element: count} -> float32 vector over `elements`, hydrogen INCLUDED.\n\n    Deliberately not TokenTable.counts_vector: that one drops H (the vocabulary has no free H token),\n    but the formula token conditions on the full formula, H and all."""\n    v = np.zeros(len(elements), np.float32)\n    ix = {e: i for i, e in enumerate(elements)}\n    for e, c in counts.items():\n        if e in ix:\n            v[ix[e]] = c\n    return v\n\n\nclass DreamsSelfiesDecoder(nn.Module):\n    """SELFIES decoder over frozen DreaMS embeddings (plan section 3.5).\n\n    memory = [W_e.emb_i + E_adduct[a_i] + E_ce[c_i]]_i  (+)  [MLP(formula_counts)]\n\n    There is no positional encoding on the memory: the spectra of a molecule are a set, and\n    test_novel_p3 asserts the permutation invariance. Input and output token embeddings are tied.\n    """\n\n    def __init__(self, n_vocab, *, pad_id, sos_id, eos_id, d_model=512, n_layers=8, n_heads=8,\n                 d_ff=2048, dropout=0.1, max_len=171, n_elements=len(ALLOWED_ELEMENTS),\n                 n_adducts=len(TEST_ADDUCTS), n_ce_bins=N_CE_BINS, d_emb=1024):\n        super().__init__()\n        self.hparams = dict(n_vocab=n_vocab, pad_id=pad_id, sos_id=sos_id, eos_id=eos_id,\n                            d_model=d_model, n_layers=n_layers, n_heads=n_heads, d_ff=d_ff,\n                            dropout=dropout, max_len=max_len, n_elements=n_elements,\n                            n_adducts=n_adducts, n_ce_bins=n_ce_bins, d_emb=d_emb)\n        self.pad_id, self.sos_id, self.eos_id = pad_id, sos_id, eos_id\n        self.d_model, self.max_len = d_model, max_len\n\n        # -- memory side --\n        self.emb_proj = nn.Linear(d_emb, d_model)\n        self.adduct_emb = nn.Embedding(n_adducts, d_model)\n        self.ce_emb = nn.Embedding(n_ce_bins, d_model)\n        self.formula_mlp = nn.Sequential(nn.Linear(n_elements, d_model), nn.GELU(),\n                                         nn.Linear(d_model, d_model))\n        self.mem_norm = nn.LayerNorm(d_model)\n\n        # -- token side --\n        self.tok_emb = nn.Embedding(n_vocab, d_model, padding_idx=pad_id)\n        self.pos_emb = nn.Embedding(max_len + 2, d_model)\n        self.drop = nn.Dropout(dropout)\n        layer = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, activation="gelu",\n                                           batch_first=True, norm_first=True)\n        self.dec = nn.TransformerDecoder(layer, n_layers, norm=nn.LayerNorm(d_model))\n        self.out_bias = nn.Parameter(torch.zeros(n_vocab))\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        nn.init.normal_(self.tok_emb.weight, std=0.02)\n        with torch.no_grad():\n            self.tok_emb.weight[self.pad_id].zero_()\n        nn.init.normal_(self.pos_emb.weight, std=0.02)\n\n    # -- memory ------------------------------------------------------------------------------- #\n\n    def build_memory(self, embs, adduct_ids, ce_ids, formula_counts, spec_mask=None):\n        """embs (B, K, d_emb) float, adduct_ids / ce_ids (B, K) long, formula_counts (B, E) float,\n        spec_mask (B, K) bool with True = a real spectrum.\n\n        -> memory (B, K+1, D), mem_pad (B, K+1) bool with True = ignore (nn.Transformer convention).\n        The formula token is last and is never padded."""\n        if embs.dim() == 2:                       # a single molecule, (K, d_emb)\n            embs, adduct_ids, ce_ids = embs[None], adduct_ids[None], ce_ids[None]\n            formula_counts = formula_counts[None]\n            spec_mask = None if spec_mask is None else spec_mask[None]\n        b, k = embs.shape[0], embs.shape[1]\n        if spec_mask is None:\n            spec_mask = torch.ones(b, k, dtype=torch.bool, device=embs.device)\n        m_spec = self.emb_proj(embs) + self.adduct_emb(adduct_ids) + self.ce_emb(ce_ids)\n        m_form = self.formula_mlp(formula_counts / FORMULA_SCALE).unsqueeze(1)\n        memory = self.mem_norm(torch.cat([m_spec, m_form], dim=1))\n        mem_pad = torch.cat([~spec_mask, torch.zeros(b, 1, dtype=torch.bool, device=embs.device)],\n                            dim=1)\n        # zero the padded spectrum tokens so they cannot leak through a fully-masked attention row\n        memory = memory.masked_fill(mem_pad.unsqueeze(-1), 0.0)\n        return memory, mem_pad\n\n    # -- decoding ----------------------------------------------------------------------------- #\n\n    def forward(self, tok_ids, memory, mem_pad=None):\n        """tok_ids (B, L) long, already prefixed with <s>. -> logits (B, L, V)."""\n        b, l = tok_ids.shape\n        assert l <= self.max_len + 2, f"sequence length {l} exceeds max_len + 2"\n        pos = torch.arange(l, device=tok_ids.device)\n        x = self.drop(self.tok_emb(tok_ids) * math.sqrt(self.d_model) + self.pos_emb(pos))\n        causal = torch.triu(torch.ones(l, l, dtype=torch.bool, device=tok_ids.device), diagonal=1)\n        h = self.dec(x, memory, tgt_mask=causal, memory_key_padding_mask=mem_pad)\n        return h @ self.tok_emb.weight.T + self.out_bias\n\n    def next_logits(self, tok_ids, memory, mem_pad=None):\n        """Logits for the token after the last position of `tok_ids`. (B, V).\n\n        Recomputes the whole prefix each call: no KV cache. At this model size that is the simpler\n        correct thing; a cache is a P4 optimisation (search_plan.md v1.1 section 0.5)."""\n        return self.forward(tok_ids, memory, mem_pad)[:, -1, :]\n\n\n# --------------------------------------------------------------------------------------------- #\n# Search: shared scaffolding\n# --------------------------------------------------------------------------------------------- #\n\nNEG_INF = -1e9\n\n\ndef _expand_memory(memory, mem_pad, n):\n    """A single molecule\'s memory -> n identical rows, so a whole beam is one forward pass."""\n    if memory.shape[0] == n:\n        return memory, mem_pad\n    assert memory.shape[0] == 1, f"expected a single-molecule memory, got {memory.shape[0]}"\n    pad = None if mem_pad is None else mem_pad.expand(n, -1)\n    return memory.expand(n, -1, -1), pad\n\n\ndef _masks_to_tensor(trackers, device):\n    return torch.from_numpy(np.stack([t.legal_mask() for t in trackers])).to(device)\n\n\ndef _logprobs(model, tok_ids, memory, mem_pad, trackers):\n    """Masked log-probabilities of the next token. Illegal tokens get NEG_INF, so a beam can never\n    step outside the formula. Rows with no legal token at all are reported as `dead`."""\n    logits = model.next_logits(tok_ids, memory, mem_pad).float()\n    legal = _masks_to_tensor(trackers, logits.device)\n    dead = ~legal.any(dim=1)\n    logits = logits.masked_fill(~legal, NEG_INF)\n    return torch.log_softmax(logits, dim=-1), legal, dead\n\n\ndef _leaf(vocab, ids, target_counts):\n    """A finished token sequence -> (smiles, ik14), or None if it fails the plan section 3.6 filter.\n\n    The tracker already guarantees the heavy-atom counts; what is checked here is everything it\n    cannot know: that the string decodes and sanitises, that it is one component, and that the\n    FULL formula including hydrogen matches. Hydrogen is only fixed once SELFIES has filled the\n    valences, which is why this check does real work."""\n    Chem = _chem()\n    smi = sf.decoder(vocab.decode(ids))\n    if not smi:\n        return None\n    m = Chem.MolFromSmiles(smi)\n    if m is None or len(Chem.GetMolFrags(m)) > 1:\n        return None\n    if mol_formula(m) != target_counts:\n        return None\n    can = Chem.MolToSmiles(m)\n    k = Chem.MolToInchiKey(m)\n    return (can, k[:14]) if k else None\n\n\nFUNNEL_KEYS = ("sequences", "eos", "valid", "unique", "dead", "truncated")\n\n\ndef new_funnel():\n    """The G3 funnel, counted by the searches themselves (gate_report.md -> G3, recorded-not-gating).\n\n    sequences  rollouts / beams started\n    eos        sequences that reached </s>, i.e. the heavy-atom formula was satisfied exactly\n    valid      of those, the ones that decode, sanitise, are one component and match the FULL\n               formula including hydrogen\n    unique     distinct ik14 among the valid ones\n    dead       sequences abandoned because no token was legal (valence ran out)\n    truncated  sequences still unfinished at MAX_LEN\n    """\n    return dict.fromkeys(FUNNEL_KEYS, 0)\n\n\ndef _collect(out, vocab, ids, logp, target_counts, fn=None):\n    """Add a finished sequence to `out`, an ik14 -> (smiles, logp) dict keeping the best logp."""\n    if fn is not None:\n        fn["eos"] += 1\n    leaf = _leaf(vocab, ids, target_counts)\n    if leaf is None:\n        return False\n    smi, key = leaf\n    if fn is not None:\n        fn["valid"] += 1\n    prev = out.get(key)\n    if prev is None or logp > prev[1]:\n        out[key] = (smi, logp)\n    return True\n\n\ndef _prep_search(model, memory, mem_pad, formula, table):\n    """Shared entry: canonical target counts, the root tracker, and the <s> row."""\n    counts = parse_formula(formula) if isinstance(formula, str) else dict(formula)\n    counts = {e: int(c) for e, c in counts.items() if c}\n    root = Tracker(table, counts)\n    assert root.missing == 0, f"formula {formula} has elements outside the vocabulary"\n    device = memory.device\n    sos = torch.full((1, 1), model.sos_id, dtype=torch.long, device=device)\n    return counts, root, sos\n\n\n# --------------------------------------------------------------------------------------------- #\n# Arm A: prefix-seeded batched sampling  (search_plan.md v1.1 Method 1, corrected)\n# --------------------------------------------------------------------------------------------- #\n\ndef _seed_prefixes(model, memory, mem_pad, table, root, sos, l_seed, k_roots, width, prefix_cap):\n    """Beam-expand to depth `l_seed`, then keep `k_roots` prefixes with a spread of scaffolds.\n\n    Seeding at depth 1 would be pointless: at derivation state 0 the mask admits only atom tokens\n    ([Branch*] needs state > 1, [Ring*] needs state > 0) and next_atom_state returns order 0 for\n    every bond prefix, so [C] / [=C] / [#C] are degenerate. The distinct roots at depth 1 are just\n    the elements of the formula. Ring and branch structure -- the scaffold -- first appears a few\n    tokens in, which is why the default is l_seed = 4.\n\n    `prefix_cap` bounds how many kept prefixes may share their first two tokens, so the K roots\n    cannot all collapse into one lineage. The cap is relaxed if too few prefixes survive it."""\n    beam = [(sos, root, 0.0)]                      # (ids (1, L), tracker, logp)\n    for _ in range(l_seed):\n        ids = torch.cat([b[0] for b in beam], dim=0)\n        trackers = [b[1] for b in beam]\n        mem, pad = _expand_memory(memory, mem_pad, len(beam))\n        lp, legal, dead = _logprobs(model, ids, mem, pad, trackers)\n        cand = []\n        for i, (bi, t, base) in enumerate(beam):\n            if dead[i]:\n                continue\n            for tok in torch.nonzero(legal[i], as_tuple=False).flatten().tolist():\n                if tok == model.eos_id:            # a complete molecule this early is not a seed\n                    continue\n                cand.append((base + float(lp[i, tok]), i, tok))\n        if not cand:\n            break\n        cand.sort(key=lambda c: -c[0])\n        nxt = []\n        for score, i, tok in cand[:width]:\n            bi, t, _ = beam[i]\n            step = torch.tensor([[tok]], dtype=torch.long, device=bi.device)\n            nxt.append((torch.cat([bi, step], dim=1), t.push(tok), score))\n        beam = nxt\n    beam.sort(key=lambda b: -b[2])\n    for cap in (prefix_cap, len(beam)):            # relax the spread rule rather than return fewer\n        seen, kept = {}, []\n        for b in beam:\n            sig = tuple(b[0][0, 1:3].tolist())\n            if seen.get(sig, 0) >= cap:\n                continue\n            seen[sig] = seen.get(sig, 0) + 1\n            kept.append(b)\n            if len(kept) == k_roots:\n                break\n        if len(kept) >= min(k_roots, len(beam)):\n            return kept\n    return kept\n\n\ndef _nucleus(lp, top_p):\n    """Zero out the tail of each row beyond cumulative probability `top_p` (log space in, log out)."""\n    srt, idx = torch.sort(lp, dim=-1, descending=True)\n    keep = torch.cumsum(srt.exp(), dim=-1) - srt.exp() < top_p\n    keep[:, 0] = True                              # never drop the mode\n    drop = torch.zeros_like(keep).scatter(1, idx, ~keep)\n    return torch.log_softmax(lp.masked_fill(drop, NEG_INF), dim=-1)\n\n\n@torch.no_grad()\ndef search_prefix_seeded(model, memory, formula, table, *, mem_pad=None, l_seed=4, k_roots=8,\n                         m_rollout=32, temperature=0.7, top_p=0.90, max_len=None, seed=None,\n                         stats=None):\n    """P3 bake-off arm A. -> [(smiles, logp)], ik14-deduped, best logp per ik14.\n\n    Pass `stats` as a dict to have the funnel counts written into it (see new_funnel)."""\n    model.eval()\n    max_len = max_len or model.max_len\n    counts, root, sos = _prep_search(model, memory, mem_pad, formula, table)\n    gen = None\n    if seed is not None:\n        gen = torch.Generator(device=memory.device).manual_seed(int(seed))\n    prefixes = _seed_prefixes(model, memory, mem_pad, table, root, sos, l_seed, k_roots,\n                              width=max(k_roots * 8, 32), prefix_cap=2)\n    out = {}\n    fn = stats if stats is not None else None\n    if fn is not None:\n        fn.update({k: fn.get(k, 0) for k in FUNNEL_KEYS})\n        fn["roots"] = fn.get("roots", 0) + len(prefixes)\n    if not prefixes:\n        return []\n    if fn is not None:\n        fn["sequences"] += len(prefixes) * m_rollout\n    ids = torch.cat([p[0].repeat(m_rollout, 1) for p in prefixes], dim=0)\n    trackers = [p[1]._copy() for p in prefixes for _ in range(m_rollout)]\n    logp = [p[2] for p in prefixes for _ in range(m_rollout)]\n    alive = list(range(ids.shape[0]))\n    while alive and ids.shape[1] < max_len + 1:\n        act = [trackers[i] for i in alive]\n        mem, pad = _expand_memory(memory, mem_pad, len(alive))\n        lp, legal, dead = _logprobs(model, ids[alive], mem, pad, act)\n        lp = _nucleus(lp / max(temperature, 1e-6), top_p)\n        tok = torch.multinomial(lp.exp(), 1, generator=gen).flatten()\n        step = torch.full((ids.shape[0], 1), model.pad_id, dtype=torch.long, device=ids.device)\n        step[alive] = tok[:, None]\n        ids = torch.cat([ids, step], dim=1)\n        nxt = []\n        for j, i in enumerate(alive):\n            if dead[j]:\n                if fn is not None:\n                    fn["dead"] += 1\n                continue\n            t = int(tok[j])\n            logp[i] += float(lp[j, t])\n            if t == model.eos_id:\n                _collect(out, table.vocab, ids[i, 1:-1].tolist(), logp[i], counts, fn)\n                continue\n            trackers[i] = trackers[i].push(t)\n            nxt.append(i)\n        alive = nxt\n    if fn is not None:\n        fn["truncated"] += len(alive)\n        fn["unique"] += len(out)\n    return sorted(((s, lg) for s, lg in out.values()), key=lambda c: -c[1])\n\n\n# --------------------------------------------------------------------------------------------- #\n# Arms B and C: masked beam search, plain and cross-beam diversified\n# --------------------------------------------------------------------------------------------- #\n\n@torch.no_grad()\ndef search_masked_beam(model, memory, formula, table, *, mem_pad=None, beam=256, max_len=None,\n                       alpha=0.0, max_leaves=None, stats=None):\n    """P3 bake-off arm B (alpha = 0) and arm C (alpha > 0). -> [(smiles, logp)], ik14-deduped.\n\n    With alpha > 0 the score used for *selection* is penalised by how often each ring / branch token\n    was taken across the previous step\'s beams, which spreads the beams over scaffold topologies\n    instead of letting them collapse onto one core. The returned logp is always the UNPENALISED\n    cumulative log-probability, because P4 orders novel candidates on it.\n\n    This is the non-circular reading of search_plan.md Method 3: the penalty uses the previous\n    step\'s counts, since a penalty defined by the current step\'s own choices is not well posed."""\n    model.eval()\n    max_len = max_len or model.max_len\n    max_leaves = beam * 4 if max_leaves is None else max_leaves\n    counts, root, sos = _prep_search(model, memory, mem_pad, formula, table)\n    scaffold = torch.from_numpy(table.is_branch | table.is_ring).to(memory.device)\n    live = [(sos, root, 0.0)]                      # (ids, tracker, unpenalised logp)\n    out = {}\n    fn = stats if stats is not None else None\n    if fn is not None:\n        fn.update({k: fn.get(k, 0) for k in FUNNEL_KEYS})\n    prev = torch.zeros(len(table.kind), device=memory.device)\n    for _ in range(max_len + 1):\n        if not live:\n            break\n        ids = torch.cat([b[0] for b in live], dim=0)\n        trackers = [b[1] for b in live]\n        mem, pad = _expand_memory(memory, mem_pad, len(live))\n        lp, legal, dead = _logprobs(model, ids, mem, pad, trackers)\n        pen = (alpha * (prev / max(len(live), 1)) * scaffold)[None, :] if alpha else 0.0\n        cand = []\n        for i, (bi, t, base) in enumerate(live):\n            if dead[i]:\n                if fn is not None:\n                    fn["dead"] += 1\n                continue\n            for tok in torch.nonzero(legal[i], as_tuple=False).flatten().tolist():\n                real = base + float(lp[i, tok])\n                rank = real - (float(pen[0, tok]) if alpha else 0.0)\n                cand.append((rank, real, i, tok))\n        if not cand:\n            break\n        cand.sort(key=lambda c: -c[0])\n        nxt, taken = [], torch.zeros_like(prev)\n        for _rank, real, i, tok in cand:\n            full = len(nxt) >= beam\n            if full and len(out) >= max_leaves:\n                break\n            bi, t, _ = live[i]\n            if tok == model.eos_id:\n                if len(out) < max_leaves:\n                    _collect(out, table.vocab, bi[0, 1:].tolist(), real, counts, fn)\n                continue\n            if full:\n                continue\n            step = torch.tensor([[tok]], dtype=torch.long, device=bi.device)\n            nxt.append((torch.cat([bi, step], dim=1), t.push(tok), real))\n            if fn is not None:\n                fn["sequences"] += 1\n            taken[tok] += 1\n        live, prev = nxt, taken\n    if fn is not None:\n        fn["truncated"] += len(live)\n        fn["unique"] += len(out)\n    return sorted(((s, lg) for s, lg in out.values()), key=lambda c: -c[1])\n\n\ndef search_diverse_beam(model, memory, formula, table, *, mem_pad=None, beam=256, alpha=0.6,\n                        max_len=None, max_leaves=None, stats=None):\n    """P3 bake-off arm C: search_masked_beam with the cross-beam penalty switched on."""\n    return search_masked_beam(model, memory, formula, table, mem_pad=mem_pad, beam=beam,\n                              max_len=max_len, alpha=alpha, max_leaves=max_leaves, stats=stats)\n\n\nSEARCHES = {"prefix_seeded": search_prefix_seeded,\n            "masked_beam": search_masked_beam,\n            "diverse_beam": search_diverse_beam}\n\n\n# --------------------------------------------------------------------------------------------- #\n# Selecting the 25 that are scored\n# --------------------------------------------------------------------------------------------- #\n\ndef morgan_fps(smiles, radius=2, n_bits=2048):\n    """MassSpecGym\'s fingerprint definition -- Morgan, radius 2, 2048 bits (`utils.morgan_fp`).\n\n    Their code calls `Chem.GetMorganFingerprintAsBitVect`, which does not exist on RDKit 2026.03.3\n    (the metric\'s pinned version): it moved to AllChem and is now deprecated in favour of\n    MorganGenerator. So the *definition* is reused, not the call. The generator returns bit-identical\n    fingerprints -- test_novel_p3.check_morgan asserts that against the deprecated function wherever\n    it is still reachable."""\n    Chem = _chem()\n    from rdkit import DataStructs\n    from rdkit.Chem import rdFingerprintGenerator\n    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)\n    fps = []\n    for s in smiles:\n        m = Chem.MolFromSmiles(s) if s else None\n        fps.append(None if m is None else gen.GetFingerprint(m))\n    return fps, DataStructs\n\n\ndef max_tanimoto(cands, truth_smiles):\n    """The metric behind G3: the best Tanimoto to the truth over a candidate list. 0.0 if empty."""\n    if not cands:\n        return 0.0\n    fps, DataStructs = morgan_fps([truth_smiles] + [c[0] for c in cands])\n    if fps[0] is None:\n        return 0.0\n    ok = [f for f in fps[1:] if f is not None]\n    if not ok:\n        return 0.0\n    return float(max(DataStructs.BulkTanimotoSimilarity(fps[0], ok)))\n\n\ndef select_top25(cands, mode="maxmin", k=25):\n    """Cut a candidate list to `k`. `cands` is [(smiles, logp)], already ik14-deduped.\n\n    "logp"   -- the most likely k. The honest decoder-likelihood ranking.\n    "maxmin" -- start from the most likely candidate, then repeatedly add whichever candidate is\n                furthest (in Tanimoto) from everything already chosen. G3 scores the *maximum*\n                similarity over the k, so covering chemical space beats stacking near-duplicates\n                of one mode (search_plan.md v1.1 section 0.2).\n    """\n    if mode not in ("logp", "maxmin"):\n        raise ValueError(f"unknown selection mode {mode!r}")\n    ranked = sorted(cands, key=lambda c: -c[1])\n    if mode == "logp" or len(ranked) <= k:\n        return ranked[:k]\n    fps, DataStructs = morgan_fps([c[0] for c in ranked])\n    pool = [i for i, f in enumerate(fps) if f is not None]\n    if len(pool) <= k:\n        return [ranked[i] for i in pool] or ranked[:k]\n    chosen = [pool[0]]                             # pool is logp-sorted, so this is the mode\n    rest = pool[1:]\n    best = list(DataStructs.BulkTanimotoSimilarity(fps[chosen[0]], [fps[i] for i in rest]))\n    while len(chosen) < k and rest:\n        j = min(range(len(rest)), key=lambda x: (best[x], -ranked[rest[x]][1]))\n        chosen.append(rest.pop(j))\n        best.pop(j)\n        if not rest:\n            break\n        sims = DataStructs.BulkTanimotoSimilarity(fps[chosen[-1]], [fps[i] for i in rest])\n        best = [max(b, s) for b, s in zip(best, sims)]\n    return [ranked[i] for i in chosen]\n'
SIMSIM_COMMON_SRC = '"""Shared helpers for the specs_simple_ms_simulator Phase-1 notebooks.\n\nInlined verbatim into the generated notebooks (at the SIMSIM_COMMON placeholder) by build/mk_simple_sim_grid.py,\nand imported directly by build/test_simple_sim_grid.py. Pure numpy / pandas / pyarrow: no RDKit, no\ntorch, no ms-pred. `prep` / `esim` call phase1\'s numba kernels (`_clean`, `entropy_sim`), which the\nnotebooks embed in an earlier cell; the tests inject them.\n\nLifted from build/simple_sim_gate.py (sh, sha1, formula_mass, the POOL build, to_model_ce, pb_sf, the\nworker call) and from phase1 (ADDUCTS physics, the stage-A/B window logic). The gate notebook itself is\nleft untouched.\n"""\nimport hashlib\nimport json\nimport os\nimport pickle\nimport re\nimport subprocess\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\n\nTEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",\n                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]\n\n# phase1 CFG values (spectrum cleaning and the candidate window) -- unchanged\nFLOOR, TOPK, POWER, ENTW, TOL = 0.002, 64, 1.0, True, 0.01\nPPM_WIN, PPM_CENTER, PPM_FALLBACK = 8.5, 1.45, 30.0\nPPM_EXPAND = (8.5, 20.0, 45.0, 80.0)\nTOPN = 25\n\n# --- adduct physics: phase1 cell 3, restricted to the ten test adducts (all n = 1, z = 1) -----------\n_M = dict(H=1.00782503207, N=14.0030740048, O=15.9949146196, C=12.0, Na=22.9897692809,\n          K=38.96370668, Cl=34.96885268)\n_E = 0.00054857990\n_PROTON = _M["H"] - _E\n_H2O = 2 * _M["H"] + _M["O"]\n_NH4 = _M["N"] + 4 * _M["H"]\n_FORMATE = _M["C"] + 2 * _M["H"] + 2 * _M["O"]\nADDUCT_DELTA = {\n    "[M+H]+": _PROTON, "[M+NH4]+": _NH4 - _E, "[M+Na]+": _M["Na"] - _E, "[M+K]+": _M["K"] - _E,\n    "[M-H2O+H]+": _PROTON - _H2O, "[M-2H2O+H]+": _PROTON - 2 * _H2O,\n    "[M-H]-": -_PROTON, "[M-H2O-H]-": -_PROTON - _H2O, "[M+CH2O2-H]-": _FORMATE - _PROTON,\n    "[M+Cl]-": _M["Cl"] + _E,\n}\n\n\ndef neutral_mass1(mz, adduct):\n    """ONE precursor m/z -> neutral mass (phase1 neutral_mass, test adducts only; NaN elsewhere).\n\n    Not called `neutral_mass`: the submission notebook also embeds phase1\'s vectorised neutral_mass.\n    """\n    d = ADDUCT_DELTA.get(adduct)\n    return float(mz) - d if d is not None else np.nan\n\n\ndef precursor_of(mass, adduct):\n    """neutral exact mass -> theoretical precursor m/z (the inverse of neutral_mass1)."""\n    return np.asarray(mass, dtype=np.float64) + ADDUCT_DELTA[adduct]\n\n\n# --- label formula -> exact neutral mass (as phase1 / the gate; no RDKit) ---------------------------\n_MONO = {"C": 12.0, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221, "S": 31.97207069,\n         "P": 30.97376151, "F": 18.99840320, "Cl": 34.96885271, "Br": 78.9183376, "I": 126.904468,\n         "Si": 27.9769265327, "Se": 79.9165196, "B": 11.0093055, "Na": 22.98976928, "K": 38.96370649,\n         "Fe": 55.9349375, "D": 2.0141017778, "As": 74.9215965, "Li": 7.0160034, "Sn": 119.9021947}\n_TOK = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    if not isinstance(f, str):\n        return np.nan\n    s, pos = 0.0, 0\n    for m in _TOK.finditer(f):\n        if m.start() != pos:\n            return np.nan\n        pos = m.end()\n        if m.group(1) not in _MONO:\n            return np.nan\n        s += _MONO[m.group(1)] * (int(m.group(2)) if m.group(2) else 1)\n    return s if pos == len(f) else np.nan\n\n\n# --- shell / files ------------------------------------------------------------------------------------\ndef sh(cmd, timeout=None, env=None):\n    print("$", cmd if len(cmd) < 300 else cmd[:300] + " ...", flush=True)\n    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout, env=env)\n    if r.stdout.strip():\n        print(r.stdout[-2500:])\n    if r.returncode:\n        print(r.stderr[-4000:])\n        raise RuntimeError(f"command failed ({r.returncode})")\n    return r.stdout\n\n\ndef sha1(path, chunk=1 << 20):\n    h = hashlib.sha1()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(chunk), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\n# --- ICEBERG worker call (the worker runs in its own venv; files only) -------------------------------\ndef to_model_ce(ce_ev, prec, mode):\n    return ce_ev if mode == "ev" else ce_ev * 500.0 / prec     # ms-pred\'s own ev_to_nce\n\n\ndef write_request(req, path):\n    req[["key", "smiles", "adduct", "precursor_mz", "ce", "instrument"]].to_csv(path, sep="\\t", index=False)\n\n\ndef worker_cmd(py, worker, gen, inten, inp, out, device="cuda", batch_size=128, sparse_k=100, prec_margin=2.0):\n    return [str(py), str(worker), "--gen", str(gen), "--inten", str(inten), "--inp", str(inp), "--out", str(out),\n            "--device", device, "--batch-size", str(batch_size), "--sparse-k", str(sparse_k),\n            "--prec-margin", str(prec_margin)]\n\n\ndef read_response(out, n):\n    """-> spectra [(mz, it)], ok, err, timing"""\n    z = np.load(out)\n    # read each array ONCE: NpzFile re-reads (and re-allocates) an array on every z[key] access, and a\n    # slice keeps its whole copy alive -- per-row access held n full copies (the Phase-1 Colab OOM)\n    off, mz, it = z["off"], z["mz"], z["it"]\n    assert len(off) == n + 1, f"{out}: {len(off) - 1} rows, expected {n}"\n    spectra = [(mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]) for i in range(n)]\n    return spectra, z["ok"], z["err"], json.loads(str(z["timing"]))\n\n\ndef run_worker(req, work, tag, py, worker, gen, inten, env, **kw):\n    """-> spectra, ok, err, timing, wall_s"""\n    inp, out = Path(work) / f"req_{tag}.tsv", Path(work) / f"resp_{tag}.npz"\n    write_request(req, inp)\n    t = time.time()\n    r = subprocess.run(worker_cmd(py, worker, gen, inten, inp, out, **kw), text=True, capture_output=True,\n                       env=env, timeout=12 * 3600)\n    if r.returncode:\n        print(r.stdout[-2000:], r.stderr[-4000:])\n        raise RuntimeError(f"worker failed on {tag}")\n    spectra, ok, err, timing = read_response(out, len(req))\n    return spectra, ok, err, timing, time.time() - t\n\n\ndef err_counts(err):\n    return pd.Series([str(e).split(":")[0] for e in err if e]).value_counts().to_dict()\n\n\n# --- spectra -----------------------------------------------------------------------------------------\ndef prep(mz, it, prec):\n    """phase1 cleaning after the precursor + 2 Da cut. Needs the phase1 kernels in globals."""\n    mz = np.asarray(mz, np.float32)\n    it = np.asarray(it, np.float32)\n    keep = mz <= prec + 2.0\n    return _clean(mz[keep], it[keep], FLOOR, TOPK, POWER, ENTW)  # noqa: F821 (phase1 kernel cell)\n\n\ndef esim(a, b):\n    if len(a[0]) == 0 or len(b[0]) == 0:\n        return 0.0\n    return float(entropy_sim(a[0], a[1], b[0], b[1], TOL))  # noqa: F821 (phase1 kernel cell)\n\n\ndef sum_merge(specs, decimals=4):\n    """Merge per-CE predicted spectra the way a merged acquisition sums them.\n\n    Each input is (mz, it) with base peak 1. Peaks with the same m/z (rounded to `decimals`; the same\n    fragment formula gives the same m/z at every CE) are summed, then the result is renormalised to\n    base 1 and sorted by m/z. Empty inputs are ignored.\n    """\n    parts = [(np.asarray(m, np.float64), np.asarray(i, np.float64)) for m, i in specs if len(m)]\n    if not parts:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    mz = np.concatenate([m for m, _ in parts])\n    it = np.concatenate([i / i.max() for _, i in parts])\n    key, inv = np.unique(np.round(mz, decimals), return_inverse=True)\n    s = np.bincount(inv, weights=it)\n    m = np.bincount(inv, weights=mz) / np.bincount(inv)\n    o = np.argsort(m)\n    return m[o].astype(np.float32), (s[o] / s.max()).astype(np.float32)\n\n\ndef rank_of_truth(sims):\n    """sims[0] = truth, sims[1:] = decoys. Ties count half; a tied truth is not top-1."""\n    sims = np.asarray(sims, dtype=np.float64)\n    t, dec = sims[0], sims[1:]\n    gt, eq = int((dec > t).sum()), int((dec == t).sum())\n    return 1 + gt + eq / 2, (gt == 0 and eq == 0)\n\n\ndef pb_sf(ps, k):\n    """P(X >= k), X = sum of independent Bernoulli(p_i) -- exact Poisson-binomial by DP."""\n    d = np.zeros(len(ps) + 1)\n    d[0] = 1.0\n    for p in ps:\n        d[1:] = d[1:] * (1 - p) + d[:-1] * p\n        d[0] *= (1 - p)\n    return float(d[k:].sum())\n\n\n# --- collision-energy grid ----------------------------------------------------------------------------\ndef ce_tuple(v):\n    """collision_energy_ev cell (list / array / None) -> tuple of floats rounded to 0.1 eV."""\n    if v is None:\n        return ()\n    try:\n        return tuple(round(float(x), 1) for x in v)\n    except TypeError:\n        return ()\n\n\ndef snap_ce(ce, grid, tol):\n    """Nearest grid CE within tol (eV), else None. Ties go to the lower CE."""\n    if not grid or not np.isfinite(ce):\n        return None\n    g = min(sorted(grid), key=lambda x: abs(x - ce))\n    return float(g) if abs(g - ce) <= tol + 1e-9 else None\n\n\ndef spectrum_conditions(adduct, ces, grid, tol, covered):\n    """One spectrum -> set of (adduct, ce, on_grid) conditions that need a simulated spectrum.\n\n    Uncovered adducts need none (MetFrag-lite). A merged multi-CE spectrum needs each constituent CE,\n    which is why merges add no grid conditions. CE <= 0 constituents (a few libraries list 0) are skipped.\n    """\n    if adduct not in covered:\n        return set()\n    out = set()\n    for ce in ces:\n        if not np.isfinite(ce) or ce <= 0:\n            continue\n        g = snap_ce(ce, grid, tol)\n        out.add((adduct, g if g is not None else round(float(ce), 1), g is not None))\n    return out\n\n\ndef choose_tolerance(median_by_delta, min_sim=0.90):\n    """Largest delta (eV) whose median sim(pred@c, pred@c+delta) is >= min_sim; 0.0 if none."""\n    ok = [d for d, m in median_by_delta.items() if np.isfinite(m) and m >= min_sim]\n    return float(max(ok)) if ok else 0.0\n\n\ndef base_ce_set(ce_tuples, min_cover=0.95):\n    """Distinct single CEs covering >= min_cover of single-CE spectra, plus every merged constituent."""\n    singles = pd.Series([t[0] for t in ce_tuples if len(t) == 1 and t[0] > 0])\n    keep = set()\n    if len(singles):\n        vc = singles.value_counts()\n        cum = vc.cumsum() / vc.sum()\n        keep = set(vc.index[: int((cum < min_cover).sum()) + 1])\n    for t in ce_tuples:\n        if len(t) > 1:\n            keep |= {c for c in t if c > 0}\n    return sorted(float(c) for c in keep)\n\n\n# --- candidate pool and windows ------------------------------------------------------------------------\ndef build_pool(coco_dir, train_path):\n    """COCONUT + train structures, deduplicated on ik14, sorted by exact neutral mass (as the gate).\n\n    Columns: k14, smiles, mass, has_real (the structure has >= 1 training spectrum).\n    """\n    coco_dir = Path(coco_dir)\n    cm = pickle.load(open(coco_dir / "coco_meta.pkl", "rb"))\n    cmass = np.load(coco_dir / "coco_mass.npy")\n    tr = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles", "molecular_formula"]).to_pandas()\n    tr = tr.dropna(subset=["inchikey14", "normalized_smiles"]).drop_duplicates("inchikey14")\n    fm = {f: formula_mass(f) for f in set(tr.molecular_formula.astype(object).tolist())}\n    tr_mass = np.array([fm.get(f, np.nan) for f in tr.molecular_formula.astype(object)], dtype=np.float64)\n    pool = pd.DataFrame({\n        "k14": np.concatenate([[str(k)[:14] for k in cm["keys"]], tr.inchikey14.astype(str).to_numpy()]),\n        "smiles": np.concatenate([np.asarray(cm["smiles"], dtype=object),\n                                  tr.normalized_smiles.astype(object).to_numpy()]),\n        "mass": np.concatenate([cmass.astype(np.float64), tr_mass]),\n    })\n    n_raw = len(pool)\n    pool = (pool[np.isfinite(pool.mass) & pool.smiles.notna()].drop_duplicates("k14")\n            .sort_values("mass", kind="stable").reset_index(drop=True))\n    pool["has_real"] = pool.k14.isin(set(tr.inchikey14.astype(str)))\n    print(f"pool {n_raw:,} rows -> {len(pool):,} unique ik14 with a finite mass "\n          f"(COCONUT {len(cmass):,}, train {len(tr):,})")\n    return pool\n\n\ndef window(pm, target, ppm):\n    """(lo, hi) index range of the mass-sorted pool within +-ppm of target (phase1 CandidatePool.window)."""\n    lo = int(np.searchsorted(pm, target * (1 - ppm / 1e6), "left"))\n    hi = int(np.searchsorted(pm, target * (1 + ppm / 1e6), "right"))\n    return lo, hi\n\n\ndef scored_range(pm, target, win=PPM_WIN, fallback=PPM_FALLBACK, expand=PPM_EXPAND, topn=TOPN):\n    """Pool range whose candidates phase1\'s stages A + B actually score (stage C only mass-sorts).\n\n    Stage A: the +-win window, or +-fallback if that is empty. Stage B: while fewer than topn candidates\n    (pool keys are unique), every wider tier\'s new candidates are scored. The windows are nested, so the\n    scored set is one contiguous range.\n    """\n    lo, hi = window(pm, target, win)\n    if hi == lo:\n        lo, hi = window(pm, target, fallback)\n    if hi - lo < topn:\n        for ppm in expand[1:]:\n            a, b = window(pm, target, ppm)\n            lo, hi = min(lo, a), max(hi, b)\n            if hi - lo >= topn:\n                break\n    return lo, hi\n\n\ndef query_target(precs, adducts):\n    """Molecule -> centred neutral target mass (phase1: median over spectra, divided by 1 + PPM_CENTER)."""\n    nms = np.array([neutral_mass1(p, a) for p, a in zip(precs, adducts)], dtype=np.float64)\n    nms = nms[np.isfinite(nms)]\n    return float(np.median(nms)) / (1.0 + PPM_CENTER / 1e6) if len(nms) else np.nan\n\n\n# --- projection and decision ---------------------------------------------------------------------------\ndef on_the_fly(conds, precomputed_adducts):\n    """Conditions left for Kaggle: off-grid ones, and on-grid ones whose adduct was not precomputed."""\n    return [c for c in conds if not (c[2] and c[0] in precomputed_adducts)]\n\n\ndef bootstrap_total(per_mol, n_mol=400, n_boot=2000, seed=0):\n    """Total over n_mol molecules resampled from per_mol -> dict(mean, p50, p95)."""\n    x = np.asarray(per_mol, dtype=np.float64)\n    if len(x) == 0:\n        return dict(mean=0.0, p50=0.0, p95=0.0)\n    rng = np.random.default_rng(seed)\n    tot = x[rng.integers(0, len(x), size=(n_boot, n_mol))].sum(1)\n    return dict(mean=float(x.mean() * n_mol), p50=float(np.percentile(tot, 50)), p95=float(np.percentile(tot, 95)))\n\n\ndef kaggle_hours(load, rate, n_mol, t_base_s, fixed_s, simlib_gb, load_s_per_gb, worker_calls, startup_s):\n    """Projected Kaggle wall time (h): fixed setup + per-molecule base cost + simlib load + on-the-fly sims."""\n    sim_s = (load / rate if load > 0 else 0.0) + (worker_calls * startup_s if load > 0 else 0.0)\n    return (fixed_s + n_mol * t_base_s + simlib_gb * load_s_per_gb + sim_s) / 3600.0\n\n\ndef decide(options, budget_h):\n    """Cheapest option (colab_h, then name) whose kaggle_h <= budget_h; None if none fits (-> STOP)."""\n    fit = [o for o in options if o["kaggle_h"] <= budget_h]\n    return min(fit, key=lambda o: (o["colab_h"], o["name"])) if fit else None\n\n\n# --- submission (Phase 4, option A: every covered condition simulated on the fly) ---------------------\nMERGE_RULES = ("max", "sum", "mid")\n\n\ndef spectrum_ces(ces, rule, default):\n    """One query spectrum\'s CE list -> (CEs to simulate, defaulted).\n\n    CE <= 0 constituents are dropped (a few lists start at 0). `max` / `sum` simulate every constituent;\n    `mid` simulates the mean of the list once. No usable CE -> [default], defaulted = True.\n    """\n    assert rule in MERGE_RULES, rule\n    v = [round(float(c), 1) for c in ces if np.isfinite(c) and c > 0]\n    if not v:\n        return [float(default)], True\n    if rule == "mid":\n        return [round(float(np.mean(v)), 1)], False\n    return sorted(set(v)), False\n\n\ndef sim_priority(lv, fr, ppm):\n    """Per-candidate simulation priority (0 = first): lv desc, fr desc, |ppm| asc -- the watchdog\'s cap order."""\n    o = np.lexsort((np.asarray(ppm, np.float64), -np.asarray(fr, np.float64), -np.asarray(lv, np.float64)))\n    r = np.empty(len(o), np.int64)\n    r[o] = np.arange(len(o))\n    return r\n\n\nREQ_COLS = ["key", "smiles", "adduct", "precursor_mz", "ce_ev", "prio", "ce", "instrument"]\n\n\ndef build_requests(parts, instrument="QTOF", ce_mode="ev"):\n    """Molecule parts -> one worker request per distinct (key, adduct, ce_ev), sorted by priority.\n\n    Each part is a dict with arrays key / smiles / mass / prio (one entry per candidate to simulate) and\n    conds, a list of (adduct, ce_ev). The precursor is the candidate\'s theoretical m/z (exact mass +\n    adduct delta, as the G2b gate), so a request depends on the candidate alone and is shared by every\n    spectrum and molecule that needs it; a shared request keeps its best (lowest) priority.\n    """\n    frames = []\n    for p in parts:\n        if not len(p["key"]) or not p["conds"]:\n            continue\n        mass = np.asarray(p["mass"], np.float64)\n        for adduct, ce in p["conds"]:\n            frames.append(pd.DataFrame({"key": np.asarray(p["key"], dtype=object),\n                                        "smiles": np.asarray(p["smiles"], dtype=object), "adduct": adduct,\n                                        "precursor_mz": precursor_of(mass, adduct), "ce_ev": float(ce),\n                                        "prio": np.asarray(p["prio"], np.int64)}))\n    if not frames:\n        return pd.DataFrame({c: [] for c in REQ_COLS})\n    req = (pd.concat(frames, ignore_index=True)\n           .sort_values(["prio", "key", "adduct", "ce_ev"], kind="stable")\n           .drop_duplicates(["key", "adduct", "ce_ev"], keep="first").reset_index(drop=True))\n    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), ce_mode)\n    req["instrument"] = instrument\n    return req[REQ_COLS]\n\n\ndef rows_that_fit(remaining_s, rate, startup_s):\n    """How many worker rows one more call can take before the deadline (0 if none)."""\n    return max(0, int((remaining_s - startup_s) * rate))\n\n\ndef spectrum_sv(q, sims, rule):\n    """Simulated-library score of ONE prepped query spectrum against one candidate.\n\n    sims: the candidate\'s raw predictions (mz, it, prec) at each of the spectrum\'s CEs, None where it was\n    not simulated (failed, capped). max / mid: best esim over the CEs; sum: esim against the renormalised\n    sum of the available CEs (a merged acquisition). Nothing available -> 0.\n    """\n    have = [s for s in sims if s is not None and len(s[0])]\n    if not have or len(q[0]) == 0:\n        return 0.0\n    if rule == "sum":\n        mz, it = sum_merge([(m, i) for m, i, _ in have])\n        return esim(q, prep(mz, it, have[0][2]))\n    return max(esim(q, prep(m, i, p)) for m, i, p in have)\n\n\ndef rank_two_tier(lv, sv, fr, ppm, tau):\n    """Order (indices) of one candidate set.\n\n    Tier 1 = real-library match lv >= tau, by lv desc (then sv, fr, |ppm|). Tier 2 = the rest, by sv desc,\n    fr desc, |ppm| asc. A missing sv / fr (NaN) counts 0; a missing ppm sorts last.\n    """\n    lv, sv, fr = (np.nan_to_num(np.asarray(x, np.float64), nan=0.0) for x in (lv, sv, fr))\n    ppm = np.nan_to_num(np.asarray(ppm, np.float64), nan=np.inf)\n    t1, t2 = np.flatnonzero(lv >= tau), np.flatnonzero(lv < tau)\n    o1 = t1[np.lexsort((ppm[t1], -fr[t1], -sv[t1], -lv[t1]))]\n    o2 = t2[np.lexsort((ppm[t2], -fr[t2], -sv[t2]))]\n    return np.concatenate([o1, o2]).astype(np.int64)\n'
NOVEL_COHORT_SRC = '"""specs_novel P1 helpers: honest benchmark cohorts and the decoder training split.\n\nThe definitions are pre-registered in specs_novel/gate_report.md -> G1 (design: phase1_plan.md §3-§4).\nnumpy / pandas plus simsim_common; RDKit and selfies are imported lazily (structure pass only).\nEmbedded verbatim into casmi_novel_p1_colab.ipynb by build/mk_novel_p0.py and imported directly by\nbuild/test_novel_p1.py.\n"""\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nimport simsim_common as ssc\n\nNP_LIBS = ("riken", "mona", "msdial", "massbank", "gnps", "enveda-np-examples", "masaryk")\nTIMSTOF_LIB = "enveda-np-examples"\nDREAMS_FT_LIB, DREAMS_FT_ADDUCT = "mona", "[M+H]+"   # the DreaMS head\'s fine-tuning data (leakage flag)\nMASS_LO, MASS_HI = 157.0, 1159.0\nPPM_FLOOR, ROUND_TOL_DA = 10.0, 0.0075\nGMAX_CHOICES = (200, 100, 50, 25)\nHELDOUT_BUDGET = 0.07\nDEC_VAL_SHARE = 0.03\nN_C3, N_C2 = 600, 400\nMAX_SPEC = 16\nSEED = 20260929\nSPLITS = ("train", "dec_val", "heldout_c3", "heldout_c2", "heldout_ts")\n\n\n# --------------------------------------------------------------------------------------------- #\n# Per structure (RDKit, lazily)\n# --------------------------------------------------------------------------------------------- #\n\ndef scaffold_key(smiles, ik14):\n    """Murcko scaffold canonical SMILES of the stereo/isotope-stripped structure.\n    Empty scaffold -> \'acyclic:<ik14>\' (a group of one); unparsable -> \'unparsable:<ik14>\'."""\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.Scaffolds import MurckoScaffold\n    RDLogger.DisableLog("rdApp.*")\n    m = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None\n    if m is None:\n        return f"unparsable:{ik14}"\n    Chem.RemoveStereochemistry(m)\n    for a in m.GetAtoms():\n        a.SetIsotope(0)\n    try:\n        s = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m))\n    except Exception:\n        return f"unparsable:{ik14}"\n    return s if s else f"acyclic:{ik14}"\n\n\n_FTOK = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_counts(f):\n    """\'C6H12O6\' -> {\'C\': 6, \'H\': 12, \'O\': 6}; None if the string is not a plain element formula."""\n    if not isinstance(f, str) or not f:\n        return None\n    out, pos = {}, 0\n    for m in _FTOK.finditer(f):\n        if m.start() != pos:\n            return None\n        pos = m.end()\n        out[m.group(1)] = out.get(m.group(1), 0) + (int(m.group(2)) if m.group(2) else 1)\n    return out if pos == len(f) else None\n\n\ndef structure_record(args):\n    """(ik14, normalized_smiles, molecular_formula, max_len) -> dict for one corpus structure.\n\n    prep via novel_decoder.prep_structure; `in_scope` = prep ok and SELFIES length <= max_len;\n    `formula_mismatch` = the RDKit formula of the prepped structure differs from the label formula\n    (None when prep did not produce a formula)."""\n    import selfies as sf\n    import novel_decoder as nd\n    ik, smi, label_formula, max_len = args\n    status, can, selfies, counts = nd.prep_structure(smi)\n    ntok = sf.len_selfies(selfies) if selfies else None\n    mismatch = None\n    if counts is not None:\n        lab = formula_counts(label_formula)\n        mismatch = lab is None or {k: v for k, v in lab.items() if v} != {k: v for k, v in counts.items() if v}\n    return dict(inchikey14=ik, status=status, can=can, selfies=selfies, ntok=ntok,\n                in_scope=bool(status == "ok" and ntok is not None and ntok <= max_len),\n                formula_mismatch=mismatch, scaffold_key=scaffold_key(smi, ik))\n\n\n# --------------------------------------------------------------------------------------------- #\n# Per row: label quality\n# --------------------------------------------------------------------------------------------- #\n\ndef label_ok(precursor_mz, adduct, formula):\n    """Vectorised G1 label-quality filter -> (ok bool array, neutral mass, truth mass).\n\n    ok = |neutral - truth| <= max(PPM_FLOOR ppm of truth, ROUND_TOL_DA) and neutral in [MASS_LO, MASS_HI].\n    Adducts outside the ten test adducts and unparsable formulas are not ok."""\n    mz = np.asarray(precursor_mz, np.float64)\n    delta = pd.Series(np.asarray(adduct, object)).map(ssc.ADDUCT_DELTA).to_numpy(np.float64)\n    f = pd.Series(np.asarray(formula, object))\n    uf = f.dropna().unique()\n    fm = f.map(dict(zip(uf, (ssc.formula_mass(x) for x in uf)))).to_numpy(np.float64)\n    neutral = mz - delta\n    tol = np.maximum(PPM_FLOOR * 1e-6 * fm, ROUND_TOL_DA)\n    with np.errstate(invalid="ignore"):\n        ok = (np.abs(neutral - fm) <= tol) & (neutral >= MASS_LO) & (neutral <= MASS_HI)\n    return ok & np.isfinite(neutral) & np.isfinite(fm), neutral, fm\n\n\n# --------------------------------------------------------------------------------------------- #\n# Per molecule: eligibility, group sizes, G_MAX\n# --------------------------------------------------------------------------------------------- #\n\ndef eligible_molecules(rows, msg_ik14):\n    """rows: DataFrame with inchikey14, ingest_lib, ok. -> one row per eligible ik14:\n    inchikey14, src_lib, timstof, n_ok (quality-OK rows in src_lib).\n\n    Eligible: ik14 not in msg_ik14 and >= 1 quality-OK row in an NP library. src_lib is the timsTOF\n    library when the molecule has quality-OK rows there, else its NP library with the most\n    quality-OK rows (ties -> the name that sorts first)."""\n    msg = set(np.asarray(msg_ik14).astype(str))\n    r = rows.loc[rows.ok.astype(bool) & rows.ingest_lib.isin(NP_LIBS), ["inchikey14", "ingest_lib"]]\n    r = r[~r.inchikey14.astype(str).isin(msg)]\n    cnt = r.groupby(["inchikey14", "ingest_lib"]).size().rename("n_ok").reset_index()\n    cnt["is_ts"] = cnt.ingest_lib == TIMSTOF_LIB\n    cnt = cnt.sort_values(["inchikey14", "is_ts", "n_ok", "ingest_lib"], ascending=[True, False, False, True])\n    out = cnt.drop_duplicates("inchikey14").rename(columns={"ingest_lib": "src_lib", "is_ts": "timstof"})\n    return out[["inchikey14", "src_lib", "timstof", "n_ok"]].reset_index(drop=True)\n\n\ndef group_sizes(scaf):\n    """scaf: Series ik14 -> scaffold_key (one entry per corpus ik14). -> Series ik14 -> group size."""\n    return scaf.map(scaf.value_counts()).astype(np.int64)\n\n\ndef choose_gmax(elig_group_size, n_universe, n_c3=N_C3, n_c2=N_C2, choices=GMAX_CHOICES,\n                budget=HELDOUT_BUDGET, dec_val=DEC_VAL_SHARE):\n    """Largest G in choices with n_c3 * m(G) + n_c2 + dec_val * U <= budget * U, where m(G) is the mean\n    group size of eligible molecules with group size <= G. -> (G, met, {G: m(G)}); if none meets the\n    budget, the smallest G with met = False."""\n    gs = np.asarray(elig_group_size, np.float64)\n    means = {int(g): (float(gs[gs <= g].mean()) if (gs <= g).any() else 0.0) for g in choices}\n    cap = budget * n_universe - n_c2 - dec_val * n_universe\n    for g in sorted(choices, reverse=True):\n        if n_c3 * means[g] <= cap:\n            return int(g), True, means\n    return int(min(choices)), False, means\n\n\n# --------------------------------------------------------------------------------------------- #\n# Cohorts\n# --------------------------------------------------------------------------------------------- #\n\ndef _draw(keys, n, rng):\n    a = np.array(sorted(keys), dtype=object)\n    if len(a) <= n:\n        return list(a)\n    return list(a[np.sort(rng.choice(len(a), size=n, replace=False))])\n\n\ndef sample_cohorts(elig, gmax, seed=SEED, n_c3=N_C3, n_c2=N_C2):\n    """elig: eligible_molecules() plus group_size and scaffold_key. -> (pseudo_c3 ik14s, c2_displace ik14s).\n\n    pseudo_c3: every eligible timsTOF molecule with group_size <= gmax (a uniform n_c3 of them if\n    more), then a uniform draw from the other eligible molecules with group_size <= gmax to n_c3.\n    c2_displace: a uniform n_c2 of the eligible molecules in no pseudo_c3 scaffold group (no cap)."""\n    rng = np.random.default_rng(seed)\n    cap = elig[elig.group_size <= gmax]\n    ts = set(cap.inchikey14[cap.timstof.astype(bool)])\n    c3 = _draw(ts, n_c3, rng)\n    rest = set(cap.inchikey14) - set(c3)\n    c3 += _draw(rest, n_c3 - len(c3), rng)\n    c3_scaf = set(elig.set_index("inchikey14").scaffold_key.loc[c3])\n    c2_pool = set(elig.inchikey14[~elig.scaffold_key.isin(c3_scaf)]) - set(c3)\n    c2 = _draw(c2_pool, n_c2, rng)\n    return sorted(c3), sorted(c2)\n\n\ndef sample_ts_dec(rows, msg_ik14, scaf, c3, c2, gmax):\n    """A-G1.1: the decoder-only timsTOF cohort. -> (sorted ts_dec ik14s, funnel dict).\n\n    Candidates: ik14 in msg_ik14 (ICEBERG has seen them) with >= 1 quality-OK timsTOF row. Excluded:\n    members of a pseudo_c3 scaffold group, c2_displace structures, members of a group holding a\n    c2_displace structure, and group size > gmax. Every remaining candidate is taken (no draw)."""\n    msg = set(np.asarray(msg_ik14).astype(str))\n    r = rows.loc[rows.ok.astype(bool) & (rows.ingest_lib == TIMSTOF_LIB), "inchikey14"].astype(str)\n    cand = set(r) & msg\n    size = scaf.value_counts()\n    c3_scaf, c2_scaf = set(scaf.loc[list(c3)]), set(scaf.loc[list(c2)])\n    after = {k for k in cand if scaf[k] not in c3_scaf and scaf[k] not in c2_scaf and k not in set(c2)}\n    ts = sorted(k for k in after if size[scaf[k]] <= gmax)\n    return ts, dict(candidates=len(cand), after_cohort_exclusions=len(after), within_gmax=len(ts))\n\n\ndef draw_n(spec_counts, rng, cap=MAX_SPEC):\n    """Spectra per molecule, drawn from the test\'s per-molecule counts, capped at `cap`."""\n    return int(min(cap, rng.choice(np.asarray(spec_counts))))\n\n\ndef pick_spectra(g, n, rng):\n    """g: candidate rows of one molecule (src_lib, quality-OK) with columns row, adduct, ce (NaN =\n    unknown). -> min(n, len(g)) row ids: CE-known rows first -- distinct (adduct, CE rounded to 0.1)\n    pairs, then the other CE-known rows -- and CE-unknown rows only to fill."""\n    g = g.sample(frac=1.0, random_state=int(rng.integers(1 << 30)))\n    ce = g.ce.to_numpy(np.float64)\n    known = g[np.isfinite(ce)]\n    unknown = g[~np.isfinite(ce)]\n    n = int(min(n, len(g)))\n    pick, seen = [], set()\n    for r in known.itertuples():\n        t = (r.adduct, round(float(r.ce), 1))\n        if t not in seen:\n            seen.add(t); pick.append(int(r.row))\n        if len(pick) == n:\n            return pick\n    chosen = set(pick)\n    for r in list(known.itertuples()) + list(unknown.itertuples()):\n        if len(pick) == n:\n            break\n        if int(r.row) not in chosen:\n            chosen.add(int(r.row)); pick.append(int(r.row))\n    return pick\n\n\ndef dreams_ft_risk(src_lib, adducts):\n    """True when the molecule\'s spectra come from the DreaMS head\'s fine-tuning data (MoNA [M+H]+)."""\n    return bool(src_lib == DREAMS_FT_LIB and DREAMS_FT_ADDUCT in set(adducts))\n\n\n# --------------------------------------------------------------------------------------------- #\n# Decoder training split\n# --------------------------------------------------------------------------------------------- #\n\ndef assign_splits(scaf, c3, c2, gmax, seed=SEED, dec_val=DEC_VAL_SHARE, ts=()):\n    """scaf: Series ik14 -> scaffold_key over every corpus ik14. -> Series ik14 -> split label.\n\n    heldout_c3: every member of a pseudo_c3 scaffold group; heldout_ts: every member of a ts_dec\n    scaffold group (A-G1.1); heldout_c2: the c2_displace structures; dec_val: whole random scaffold\n    groups with size <= gmax that hold no cohort structure, added in random order while the total\n    stays <= dec_val * U; train: the rest."""\n    rng = np.random.default_rng(seed + 1)\n    size = scaf.value_counts()\n    c3_scaf = set(scaf.loc[list(c3)])\n    ts_scaf = set(scaf.loc[list(ts)])\n    assert not ts_scaf & c3_scaf, "a ts_dec group is also a pseudo_c3 group"\n    split = pd.Series("train", index=scaf.index, dtype=object)\n    split[scaf.isin(c3_scaf)] = "heldout_c3"\n    split[scaf.isin(ts_scaf)] = "heldout_ts"\n    c2 = list(c2)\n    assert split.loc[c2].eq("train").all(), "a c2_displace structure is in a pseudo_c3 or ts_dec group"\n    split.loc[c2] = "heldout_c2"\n    c2_scaf = set(scaf.loc[c2])\n    busy = c3_scaf | ts_scaf | c2_scaf\n    cand = np.array(sorted(k for k, s in size.items() if s <= gmax and k not in busy),\n                    dtype=object)\n    rng.shuffle(cand)\n    target, total, take = dec_val * len(scaf), 0, []\n    for k in cand:\n        s = int(size[k])\n        if total + s <= target:\n            take.append(k); total += s\n        if total >= target - 0.5:\n            break\n    split[scaf.isin(set(take))] = "dec_val"\n    return split\n\n\ndef check_invariants(split, scaf, c3, c2, cohort_rows, row_ok, row_lib, src_lib_of, msg_ik14, ts=()):\n    """G1.4 (G1.4\' with ts_dec, A-G1.1) -> {invariant: bool}.\n\n    cohort_rows: {ik14: [corpus rows]} over every cohort; row_ok / row_lib: per corpus row;\n    src_lib_of: {ik14: src_lib}."""\n    msg = set(np.asarray(msg_ik14).astype(str))\n    c3, c2, ts = set(c3), set(c2), set(ts)\n    c3_members = set(scaf.index[scaf.isin(set(scaf.loc[list(c3)]))])\n    ts_members = set(scaf.index[scaf.isin(set(scaf.loc[list(ts)]))])\n    ok = {\n        "split_labels_valid": bool(split.isin(SPLITS).all()),\n        "split_covers_every_ik14_once": bool(split.index.is_unique and set(split.index) == set(scaf.index)),\n        "c3_groups_not_in_train_or_dec_val": bool(split.loc[list(c3_members)].eq("heldout_c3").all()),\n        "c2_not_in_train_or_dec_val": bool(split.loc[list(c2)].eq("heldout_c2").all()),\n        "cohort_rows_quality_ok_from_src_lib": all(\n            len(rs) > 0 and all(bool(row_ok[r]) and row_lib[r] == src_lib_of[ik] for r in rs)\n            for ik, rs in cohort_rows.items()),\n        "no_cohort_ik14_in_msg": not ((c3 | c2) & msg),\n        "cohorts_disjoint": not (c3 & c2),\n        "ts_groups_not_in_train_or_dec_val": bool(split.loc[list(ts_members)].eq("heldout_ts").all()),\n        "ts_disjoint_from_c3_c2": not (ts & (c3 | c2)),\n        "ts_all_in_msg": ts <= msg,\n        "ts_src_lib_timstof": all(src_lib_of[k] == TIMSTOF_LIB for k in ts),\n    }\n    return ok\n'
NOVEL_P3_DATA_SRC = '"""specs_novel P3 training data: the structure pass, the training index and the epoch sampler.\n\nPre-registered in `specs_novel/gate_report.md` -> G3. Numpy / pandas only, no torch: the notebook\nturns a batch into tensors. That keeps this module testable on its own.\n\n    build_structures      every unique ik14 -> (status, can, selfies, formula); multiprocessing\n    build_index           dec_split + corpus_meta + corpus_ce + structures -> a compact index\n    EmbeddingStore        memory-maps `_novel/emb_dbec3a0_lx/emb_*.npy` and gathers rows\n    make_batch            one batch: spectra sampled per molecule, with CE dropout\n\nThree things here are easy to get wrong and are therefore spelled out:\n\n1. **The oracle formula comes from the structure, not the label.** `prep_structure(can)` returns the\n   RDKit formula with hydrogen included; 165 corpus structures disagree with `molecular_formula`\n   (G1 finding 4). Only the structure formula is used, for the memory token, the search masks and\n   the leaf check alike.\n2. **Embedding row == corpus_meta row.** chunk = row // CHUNK, offset = row % CHUNK. Only\n   `emb_dbec3a0_lx` is a valid source; `emb_dbec3a0` and `_dreams_out/emb` were built with the old\n   `dreams_peaks` and must never be used for specs_novel (G0.1-R2).\n3. **The training set is `dec_split == "train"`.** `dec_val` and every `heldout_*` label are the\n   model\'s exclusion set, not merely an eval holdout.\n"""\nimport json\nimport multiprocessing as mp\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport novel_cohort as nc\nimport novel_decoder as nd\n\nCHUNK = 200_000                      # rows per embedding chunk file\nEMB_DIM = 1024\nMAX_LEN = 171                        # P0.3, p99.9 of corpus SELFIES token lengths\nMAX_SPEC_PER_MOL = 4                 # G3\nCE_DROPOUT = 0.3                     # G3\nADDUCT_IX = {a: i for i, a in enumerate(nd.TEST_ADDUCTS)}\nHILL = ("C", "H")                    # Hill order: C, H, then the rest alphabetically\n\n\ndef formula_str(counts):\n    """{element: count} -> a canonical Hill-order string, so a formula round-trips through parquet."""\n    rest = sorted(e for e in counts if e not in HILL and counts[e])\n    out = []\n    for e in list(HILL) + rest:\n        c = int(counts.get(e, 0))\n        if c:\n            out.append(e if c == 1 else f"{e}{c}")\n    return "".join(out)\n\n\n# --------------------------------------------------------------------------------------------- #\n# The structure pass\n# --------------------------------------------------------------------------------------------- #\n\n_VOCAB = None\n\n\ndef _init_worker(vocab_path):\n    global _VOCAB\n    _VOCAB = nd.SelfiesVocab.from_json(vocab_path)\n\n\ndef _structure(args):\n    """(ik14, smiles, label_formula, max_len) -> novel_cohort.structure_record plus the P3 fields.\n\n    P1\'s record is reused verbatim so the two phases cannot drift on what a structure *is*\n    (status, canonical SMILES, SELFIES, scaffold_key, its own `in_scope`). P3 adds only:\n\n      formula   the ORACLE formula, from the prepared structure via RDKit with hydrogen included.\n                Never `molecular_formula`: 165 corpus structures disagree with their label.\n      n_tokens / n_unk   under our vocabulary\n      in_scope  P1\'s, tightened by "every token is in the vocabulary and there is at least one".\n                A token the vocabulary lacks can never be generated, so such a molecule is not\n                reachable and must not be trained on as if it were.\n    """\n    rec = nc.structure_record(args)\n    ids = _VOCAB.encode(rec["selfies"]) if rec["selfies"] else []\n    rec["n_tokens"] = len(ids)\n    rec["n_unk"] = sum(1 for i in ids if i == _VOCAB.unk_id)\n    rec["formula"] = None\n    if rec["status"] == "ok" and rec["can"]:\n        Chem = nd._chem()\n        m = Chem.MolFromSmiles(rec["can"])\n        if m is not None:\n            rec["formula"] = formula_str(nd.mol_formula(m))\n    rec["in_scope"] = bool(rec["in_scope"] and rec["n_unk"] == 0 and rec["n_tokens"] > 0\n                           and rec["formula"])\n    return rec\n\n\ndef build_structures(ik14, smiles, label_formula, vocab_path, max_len=MAX_LEN, workers=None,\n                     chunksize=256):\n    """One row per unique ik14. Fork pool, as the P1 notebook does.\n\n    Feed it `META.drop_duplicates("inchikey14")` in corpus-row order -- the same representative row\n    per structure that P1 used, so scaffolds and splits line up."""\n    args = [(a, b, c, max_len) for a, b, c in zip(ik14, smiles, label_formula)]\n    workers = workers or os.cpu_count() or 1\n    if workers > 1:\n        ctx = mp.get_context("fork") if hasattr(os, "fork") else mp.get_context("spawn")\n        with ctx.Pool(workers, initializer=_init_worker, initargs=(str(vocab_path),)) as pool:\n            recs = pool.map(_structure, args, chunksize=chunksize)\n    else:\n        _init_worker(str(vocab_path))\n        recs = [_structure(a) for a in args]\n    return pd.DataFrame.from_records(recs)\n\n\n# --------------------------------------------------------------------------------------------- #\n# The training index\n# --------------------------------------------------------------------------------------------- #\n\nclass Index:\n    """A compact, fork-friendly training index. Ragged lists are flat arrays plus offsets, so a\n    forked worker shares them instead of copying a Python object graph.\n\n    tok_flat / tok_off   SELFIES token ids per molecule\n    row_flat / row_off   corpus rows (== embedding rows) per molecule\n    row_adduct           adduct index into nd.TEST_ADDUCTS, per row of row_flat\n    row_ce               CE bin per row of row_flat (nd.CE_UNKNOWN where the CE is unknown)\n    fvec                 (N, len(ALLOWED_ELEMENTS)) formula counts, hydrogen included\n    """\n\n    __slots__ = ("ik14", "formula", "tok_flat", "tok_off", "row_flat", "row_off", "row_adduct",\n                 "row_ce", "fvec", "meta")\n\n    def __init__(self, ik14, formula, tok_flat, tok_off, row_flat, row_off, row_adduct, row_ce,\n                 fvec, meta=None):\n        self.ik14, self.formula = ik14, formula\n        self.tok_flat, self.tok_off = tok_flat, tok_off\n        self.row_flat, self.row_off = row_flat, row_off\n        self.row_adduct, self.row_ce = row_adduct, row_ce\n        self.fvec = fvec\n        self.meta = meta or {}\n\n    def __len__(self):\n        return len(self.ik14)\n\n    def tokens(self, i):\n        return self.tok_flat[self.tok_off[i]:self.tok_off[i + 1]]\n\n    def rows(self, i):\n        return self.row_flat[self.row_off[i]:self.row_off[i + 1]]\n\n    def n_spectra(self, i):\n        return int(self.row_off[i + 1] - self.row_off[i])\n\n    def save(self, path):\n        path = Path(path)\n        np.savez(path, ik14=np.asarray(self.ik14, object), formula=np.asarray(self.formula, object),\n                 tok_flat=self.tok_flat, tok_off=self.tok_off, row_flat=self.row_flat,\n                 row_off=self.row_off, row_adduct=self.row_adduct, row_ce=self.row_ce,\n                 fvec=self.fvec, meta=np.asarray(json.dumps(self.meta), object))\n\n    @classmethod\n    def load(cls, path):\n        z = np.load(path, allow_pickle=True)\n        return cls(list(z["ik14"]), list(z["formula"]), z["tok_flat"], z["tok_off"], z["row_flat"],\n                   z["row_off"], z["row_adduct"], z["row_ce"], z["fvec"],\n                   json.loads(str(z["meta"])))\n\n\ndef row_quality(meta):\n    """Per corpus row: is it usable for training? -> bool array.\n\n    G1\'s label-quality filter, plus the requirement that the adduct is one of the ten test adducts\n    (the decoder has an embedding row per adduct and nothing else)."""\n    ok, _neutral, _truth = nc.label_ok(meta["precursor_mz"].to_numpy(),\n                                       meta["adduct"].to_numpy(object),\n                                       meta["molecular_formula"].to_numpy(object))\n    known = meta["adduct"].map(ADDUCT_IX).notna().to_numpy()\n    return ok & known\n\n\ndef build_index(meta, ce, split, structures, vocab, keep_splits=("train",)):\n    """-> Index over the molecules of `keep_splits` that are in scope and have >= 1 quality-OK row.\n\n    meta        corpus_meta.parquet, row i == embedding row i\n    ce          corpus_ce.parquet with `row` and `ce_mean`, or None\n    split       dec_split.parquet with `inchikey14` and `split`\n    structures  build_structures() output\n    vocab       SelfiesVocab -- passed explicitly, because parquet does not keep DataFrame.attrs\n    """\n    keep_ik = set(split.loc[split["split"].isin(keep_splits), "inchikey14"])\n    st = structures[structures["in_scope"] & structures["inchikey14"].isin(keep_ik)]\n    st = st.set_index("inchikey14")\n\n    ok = row_quality(meta)\n    ce_mean = pd.Series(np.nan, index=meta.index, dtype="float64")\n    if ce is not None and len(ce):\n        ce_mean.iloc[ce["row"].to_numpy()] = ce["ce_mean"].to_numpy(np.float64)\n\n    use = pd.DataFrame({"row": np.arange(len(meta), dtype=np.int64),\n                        "ik14": meta["inchikey14"].to_numpy(object),\n                        "adduct": meta["adduct"].to_numpy(object),\n                        "ce_mean": ce_mean.to_numpy()})[ok]\n    use = use[use["ik14"].isin(st.index)]\n    use = use.sort_values("row", kind="stable")\n\n    groups = use.groupby("ik14", sort=True)\n    ik14 = list(groups.groups.keys())\n    counts = groups.size().reindex(ik14).to_numpy(np.int64)\n\n    row_flat = use["row"].to_numpy(np.int64)\n    order = np.argsort(use["ik14"].to_numpy(object), kind="stable")\n    row_flat = row_flat[order]\n    adducts = use["adduct"].to_numpy(object)[order]\n    ces = use["ce_mean"].to_numpy()[order]\n    row_adduct = np.array([ADDUCT_IX[a] for a in adducts], np.int8)\n    row_ce = np.array([nd.ce_bin(c) for c in ces], np.int8)\n    row_off = np.concatenate([[0], np.cumsum(counts)]).astype(np.int64)\n\n    toks, tok_off, formula, fvec = [], [0], [], []\n    for ik in ik14:\n        rec = st.loc[ik]\n        ids = vocab.encode(rec["selfies"])\n        toks.extend(ids)\n        tok_off.append(len(toks))\n        formula.append(rec["formula"])\n        fvec.append(nd.formula_vector(nd.parse_formula(rec["formula"])))\n    meta_info = dict(n_molecules=len(ik14), n_rows=int(row_off[-1]), keep_splits=list(keep_splits),\n                     max_len=MAX_LEN, vocab_size=len(vocab))\n    return Index(ik14, formula, np.asarray(toks, np.int32), np.asarray(tok_off, np.int64),\n                 row_flat, row_off, row_adduct, row_ce,\n                 np.asarray(fvec, np.float32), meta_info)\n\n\n# --------------------------------------------------------------------------------------------- #\n# Embeddings\n# --------------------------------------------------------------------------------------------- #\n\nclass EmbeddingStore:\n    """Memory-maps the chunked DreaMS cache. Row r lives at chunk r // CHUNK, offset r % CHUNK."""\n\n    def __init__(self, directory, n_rows=None, chunk=CHUNK, dim=EMB_DIM, pattern="emb_{:04d}.npy"):\n        self.dir, self.chunk, self.dim = Path(directory), chunk, dim\n        assert "emb_dbec3a0_lx" in str(self.dir) or os.environ.get("NOVEL_ALLOW_ANY_EMB"), (\n            f"{self.dir} is not the G0.1-R2 cache; only emb_dbec3a0_lx may be used for specs_novel")\n        self.parts, i = [], 0\n        while True:\n            p = self.dir / pattern.format(i)\n            if not p.exists():\n                break\n            self.parts.append(np.load(p, mmap_mode="r"))\n            i += 1\n        assert self.parts, f"no embedding chunks under {self.dir}"\n        for j, a in enumerate(self.parts):\n            assert a.ndim == 2 and a.shape[1] == dim, (j, a.shape)\n            # row -> (chunk, offset) is only correct if every chunk but the last is exactly full.\n            # A short middle chunk would silently return the WRONG embedding for every later row,\n            # which no downstream check could catch, so it is refused here.\n            assert a.shape[0] == chunk or j == len(self.parts) - 1, (\n                f"chunk {j} has {a.shape[0]} rows, expected {chunk}: row -> chunk mapping "\n                f"would be wrong for every later row")\n        assert self.parts[-1].shape[0] <= chunk, (len(self.parts) - 1, self.parts[-1].shape)\n        self.n_rows = sum(a.shape[0] for a in self.parts)\n        if n_rows is not None:\n            assert self.n_rows == n_rows, f"cache has {self.n_rows} rows, corpus has {n_rows}"\n\n    def gather(self, rows):\n        """-> (len(rows), dim) float32. Reads each touched chunk once, in row order."""\n        rows = np.asarray(rows, np.int64)\n        out = np.empty((len(rows), self.dim), np.float32)\n        if not len(rows):\n            return out\n        ci = rows // self.chunk\n        for c in np.unique(ci):\n            sel = ci == c\n            out[sel] = self.parts[int(c)][rows[sel] % self.chunk]\n        return out\n\n\n# --------------------------------------------------------------------------------------------- #\n# Batching\n# --------------------------------------------------------------------------------------------- #\n\ndef sample_spectra(index, i, rng, max_spec=MAX_SPEC_PER_MOL):\n    """Up to `max_spec` of molecule i\'s rows, drawn without replacement each epoch. This both\n    augments the data and matches the test\'s 1-16 spectra per molecule."""\n    rows = index.rows(i)\n    if len(rows) <= max_spec:\n        return np.arange(index.row_off[i], index.row_off[i + 1])\n    off = index.row_off[i]\n    return off + rng.choice(len(rows), max_spec, replace=False)\n\n\ndef make_batch(index, mol_ids, store, rng, *, max_spec=MAX_SPEC_PER_MOL, ce_dropout=CE_DROPOUT,\n               pad_id, sos_id, eos_id):\n    """One training batch as numpy arrays.\n\n    -> dict with\n       emb        (B, K, EMB_DIM) float32     K = the batch\'s largest spectrum count\n       adduct     (B, K) int64\n       ce         (B, K) int64                CE_UNKNOWN where unknown or dropped\n       spec_mask  (B, K) bool                 True = a real spectrum\n       fvec       (B, E) float32\n       tgt        (B, L+2) int64              <s> tokens </s>, pad-filled\n       ce_known   (B, K) bool                 before dropout, for the CE-split reporting\n    """\n    sel = [sample_spectra(index, i, rng, max_spec) for i in mol_ids]\n    k = max(len(s) for s in sel)\n    b = len(mol_ids)\n    flat = np.concatenate(sel) if b else np.zeros(0, np.int64)\n    gathered = store.gather(index.row_flat[flat]) if len(flat) else np.zeros((0, store.dim), np.float32)\n\n    emb = np.zeros((b, k, store.dim), np.float32)\n    adduct = np.zeros((b, k), np.int64)\n    ce = np.full((b, k), nd.CE_UNKNOWN, np.int64)\n    ce_known = np.zeros((b, k), bool)\n    spec_mask = np.zeros((b, k), bool)\n    at = 0\n    for j, s in enumerate(sel):\n        n = len(s)\n        emb[j, :n] = gathered[at:at + n]\n        adduct[j, :n] = index.row_adduct[s]\n        raw = index.row_ce[s].astype(np.int64)\n        ce_known[j, :n] = raw != nd.CE_UNKNOWN\n        if ce_dropout > 0:\n            drop = rng.random(n) < ce_dropout\n            raw = np.where(drop, nd.CE_UNKNOWN, raw)\n        ce[j, :n] = raw\n        spec_mask[j, :n] = True\n        at += n\n\n    lens = [index.tok_off[i + 1] - index.tok_off[i] for i in mol_ids]\n    lmax = int(max(lens)) + 2\n    tgt = np.full((b, lmax), pad_id, np.int64)\n    for j, i in enumerate(mol_ids):\n        t = index.tokens(i)\n        tgt[j, 0] = sos_id\n        tgt[j, 1:1 + len(t)] = t\n        tgt[j, 1 + len(t)] = eos_id\n    return dict(emb=emb, adduct=adduct, ce=ce, spec_mask=spec_mask,\n                fvec=index.fvec[np.asarray(mol_ids, np.int64)], tgt=tgt, ce_known=ce_known)\n\n\ndef epoch_order(n, rng, batch_size, length=None, bucket=8):\n    """Molecule ids for one epoch, shuffled, then length-bucketed so a batch pads little.\n\n    Shuffle first, then sort within a window of `bucket` batches: batches stay random across epochs\n    but hold similar-length sequences, which is where the padding saving comes from."""\n    ids = rng.permutation(n)\n    if length is None:\n        return [ids[i:i + batch_size] for i in range(0, n, batch_size)]\n    out, window = [], batch_size * bucket\n    for s in range(0, n, window):\n        w = ids[s:s + window]\n        w = w[np.argsort(length[w], kind="stable")]\n        out.extend(w[i:i + batch_size] for i in range(0, len(w), batch_size))\n    rng.shuffle(out)\n    return out\n'
NOVEL_P3_EVAL_SRC = '"""specs_novel P3 evaluation helpers: cohorts, resumable generation, and the G3 metrics.\n\nPre-registered in `specs_novel/gate_report.md` -> G3. Shared by the GPU eval notebook (generation and\nthe bake-off) and the CPU null notebook (RandomDeNovo, MCES and the verdict).\n\n    build_queries      a cohort parquet or a dec_val Index -> one query per molecule\n    ChunkStore         append candidates to Drive every N molecules, with a skip list, so a\n                       disconnected generation run resumes instead of restarting\n    score_molecule     top-25 max Tanimoto and exact@k for one molecule, per selection mode\n    summarise          per-cohort medians and the delta against the null arm\n    g3_verdict         the pre-registered G3.1 / G3.2 decision\n\nTwo invariants this module enforces rather than documents:\n\n* **The oracle formula comes from the structure.** `formula` in a cohort parquet is the *label*\n  formula and 165 corpus structures disagree with it (G1 finding 4). Queries carry the RDKit formula\n  of `can`, hydrogen included, and that one formula feeds the memory token, the search masks and the\n  leaf check alike.\n* **Both arms see the same molecules and the same formula.** `score_molecule` takes one query and one\n  candidate list, so the decoder arm and the RandomDeNovo arm are scored by identical code. G3 is a\n  comparison; an asymmetry anywhere in it would make the number an artefact.\n"""\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport novel_decoder as nd\nimport novel_p3_data as p3d\n\nTOP_K = 25\nEXACT_KS = (1, 10, 25)\nSELECT_MODES = ("maxmin", "logp")\nDEC_VAL_MAX_SPEC = 16          # the test\'s cap on spectra per molecule\n\n\ndef oracle_formula(can):\n    """The formula the gates use: RDKit\'s, from the prepared structure, hydrogen included."""\n    Chem = nd._chem()\n    m = Chem.MolFromSmiles(can)\n    if m is None:\n        return None\n    return p3d.formula_str(nd.mol_formula(m))\n\n\ndef build_queries(cohort, limit=None, seed=None):\n    """A cohort parquet (pseudo_c3 / ts_dec / c2_displace) -> a list of query dicts.\n\n    Each query: molecule_id, ik14, can, formula (ORACLE), rows, adducts, ces, src_lib, timstof,\n    n_ce_known. Molecules out of scope or whose structure will not parse are dropped and reported by\n    the caller -- they are unreachable by the decoder, so scoring them as misses would blame the\n    search for a vocabulary limit."""\n    df = cohort\n    if limit is not None and len(df) > limit:\n        rng = np.random.default_rng(seed)\n        df = df.iloc[np.sort(rng.choice(len(df), limit, replace=False))]\n    out, skipped = [], []\n    for _, r in df.iterrows():\n        f = oracle_formula(r["can"])\n        if f is None or not bool(r.get("in_scope", True)):\n            skipped.append(str(r["molecule_id"]))\n            continue\n        ces = list(r["ces"]) if r["ces"] is not None else []\n        out.append(dict(molecule_id=str(r["molecule_id"]), ik14=str(r["inchikey14"]),\n                        can=str(r["can"]), formula=f,\n                        rows=np.asarray(r["rows"], np.int64),\n                        adducts=[str(a) for a in r["adducts"]],\n                        ces=[None if c is None or not np.isfinite(c) else float(c) for c in ces],\n                        src_lib=str(r.get("src_lib", "")), timstof=bool(r.get("timstof", False)),\n                        n_ce_known=int(sum(1 for c in ces if c is not None and np.isfinite(c)))))\n    return out, skipped\n\n\ndef queries_from_index(index, structures, limit=None, seed=None, max_spec=DEC_VAL_MAX_SPEC):\n    """The same query shape from a `novel_p3_data.Index` (used for the dec_val bake-off subset)."""\n    can_of = structures.set_index("inchikey14")["can"].to_dict()\n    ids = np.arange(len(index))\n    if limit is not None and len(ids) > limit:\n        rng = np.random.default_rng(seed)\n        ids = np.sort(rng.choice(len(ids), limit, replace=False))\n    out, skipped = [], []\n    for i in ids:\n        ik = index.ik14[i]\n        can = can_of.get(ik)\n        f = oracle_formula(can) if can else None\n        if f is None:\n            skipped.append(ik)\n            continue\n        sl = slice(index.row_off[i], index.row_off[i] + min(max_spec, index.n_spectra(i)))\n        ce = index.row_ce[sl]\n        out.append(dict(molecule_id=ik, ik14=ik, can=can, formula=f,\n                        rows=index.row_flat[sl].astype(np.int64),\n                        adducts=[nd.TEST_ADDUCTS[a] for a in index.row_adduct[sl]],\n                        ce_bins=ce.astype(np.int64).tolist(), ces=None,\n                        src_lib="dec_val", timstof=False,\n                        n_ce_known=int((ce != nd.CE_UNKNOWN).sum())))\n    return out, skipped\n\n\ndef query_memory(model, query, store, device, torch):\n    """-> (memory, mem_pad) for one query: its spectra plus the oracle formula token."""\n    emb = store.gather(query["rows"])\n    if "ce_bins" in query and query["ce_bins"] is not None:\n        ce = list(query["ce_bins"])\n    else:\n        ce = [nd.ce_bin(c) for c in (query["ces"] or [None] * len(query["rows"]))]\n    ce = (ce + [nd.CE_UNKNOWN] * len(emb))[:len(emb)]\n    ad = [nd.TEST_ADDUCTS.index(a) if a in nd.TEST_ADDUCTS else 0 for a in query["adducts"]]\n    ad = (ad + [0] * len(emb))[:len(emb)]\n    fv = nd.formula_vector(nd.parse_formula(query["formula"]))\n    t = lambda a, dt: torch.as_tensor(np.asarray(a)[None], dtype=dt, device=device)\n    return model.build_memory(t(emb, torch.float32), t(ad, torch.long), t(ce, torch.long),\n                              t(fv, torch.float32))\n\n\n# --------------------------------------------------------------------------------------------- #\n# Resumable generation\n# --------------------------------------------------------------------------------------------- #\n\ndef randomdenovo_smiles(rd, formula):\n    """One RandomDeNovo draw -> [smiles].\n\n    `generate_random_molecule_graphs_via_traversal` is typed `-> list[Mol]`\n    (MSGYM_novel/massspecgym/models/de_novo/random.py:746-750) and returns up to `max_top_k`\n    molecules per call, never a single Mol. Handing that list straight to `Chem.MolToSmiles` raises\n    `ArgumentError`, which a caller\'s `except Exception` silently turns into "this molecule produced\n    no candidates" -- and a null arm of exactly zero is not a baseline, it hands G3.1 a free pass at\n    delta = the treatment\'s own score. So the unpacking lives here, in the module both notebooks\n    import, and is covered by a test.\n\n    Note also that `RandomDeNovo.__init__` sets `cache_results=True` by default, so repeat calls for\n    the same formula return the identical list object: a bigger pool needs `cache_results=False`,\n    not more calls."""\n    from rdkit import Chem\n    got = rd.generate_random_molecule_graphs_via_traversal(formula)\n    if got is None:\n        return []\n    if not isinstance(got, (list, tuple)):\n        got = [got]\n    out = []\n    for g in got:\n        if g is None:\n            continue\n        try:\n            smi = g if isinstance(g, str) else Chem.MolToSmiles(g)\n        except Exception:\n            continue\n        if smi:\n            out.append(smi)\n    return out\n\n\nclass ChunkStore:\n    """Candidates flushed to Drive every `chunk` molecules, with a done-list beside them.\n\n    Generation over hundreds of molecules is the longest leg of P3, so it must survive a Colab\n    disconnect the way training does. Reloading yields the set of molecule_ids already finished; the\n    caller skips those. The parquet is only ever appended to, one file per flush, so a crash during a\n    write cannot corrupt earlier work."""\n\n    def __init__(self, directory, tag, chunk=50):\n        self.dir = Path(directory)\n        self.dir.mkdir(parents=True, exist_ok=True)\n        self.tag, self.chunk = tag, chunk\n        self.done_path = self.dir / f"done_{tag}.json"\n        self.buf, self.n_flush = [], 0\n\n    def done(self):\n        if not self.done_path.exists():\n            return set()\n        try:\n            return set(json.loads(self.done_path.read_text()))\n        except Exception:\n            return set()\n\n    def parts(self):\n        # The chunk index is anchored to digits because tags nest: a plain `_*` glob on tag\n        # "randomdenovo_pseudo_c3" also matches "cands_randomdenovo_pseudo_c3_pool_0000.parquet",\n        # which silently merged the pool null\'s 300 candidates into the gated null\'s 25.\n        return sorted(self.dir.glob(f"cands_{self.tag}_[0-9]*.parquet"))\n\n    def add(self, molecule_id, cands, funnel, seconds):\n        for rank, (smi, logp) in enumerate(cands):\n            self.buf.append(dict(molecule_id=molecule_id, rank=rank, smiles=smi, logp=float(logp)))\n        if not cands:                              # keep the molecule on the record even with 0 hits\n            self.buf.append(dict(molecule_id=molecule_id, rank=-1, smiles=None, logp=np.nan))\n        self._meta(molecule_id, funnel, seconds)\n        if len({r["molecule_id"] for r in self.buf}) >= self.chunk:\n            self.flush()\n\n    def _meta(self, molecule_id, funnel, seconds):\n        rec = dict(molecule_id=molecule_id, seconds=float(seconds), **{k: int(v) for k, v in\n                                                                      (funnel or {}).items()})\n        with open(self.dir / f"funnel_{self.tag}.jsonl", "a") as f:\n            f.write(json.dumps(rec) + "\\n")\n            f.flush()\n            os.fsync(f.fileno())\n\n    def flush(self, extra_done=()):\n        if self.buf:\n            df = pd.DataFrame(self.buf)\n            tmp = self.dir / f"cands_{self.tag}_{self.n_flush:04d}.parquet.tmp"\n            dest = self.dir / f"cands_{self.tag}_{self.n_flush:04d}.parquet"\n            while dest.exists():\n                self.n_flush += 1\n                tmp = self.dir / f"cands_{self.tag}_{self.n_flush:04d}.parquet.tmp"\n                dest = self.dir / f"cands_{self.tag}_{self.n_flush:04d}.parquet"\n            df.to_parquet(tmp, index=False)\n            os.replace(tmp, dest)\n            self.n_flush += 1\n            ids = {r["molecule_id"] for r in self.buf}\n            self.buf = []\n        else:\n            ids = set()\n        ids |= set(extra_done)\n        allids = self.done() | ids\n        tmp = Path(str(self.done_path) + ".tmp")\n        tmp.write_text(json.dumps(sorted(allids)))\n        os.replace(tmp, self.done_path)\n        return len(allids)\n\n    def load(self):\n        """-> {molecule_id: [(smiles, logp), ...]} over every flushed part, rank-ordered."""\n        parts = self.parts()\n        if not parts:\n            return {}\n        df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)\n        df = df[df["rank"] >= 0].sort_values(["molecule_id", "rank"], kind="stable")\n        return {m: list(zip(g["smiles"], g["logp"])) for m, g in df.groupby("molecule_id")}\n\n    def funnels(self):\n        p = self.dir / f"funnel_{self.tag}.jsonl"\n        if not p.exists():\n            return pd.DataFrame()\n        return pd.DataFrame([json.loads(l) for l in p.read_text().splitlines() if l.strip()])\n\n\n# --------------------------------------------------------------------------------------------- #\n# Metrics\n# --------------------------------------------------------------------------------------------- #\n\ndef score_molecule(query, cands, modes=SELECT_MODES, k=TOP_K, exact_ks=EXACT_KS):\n    """One molecule, one candidate list -> the G3 numbers, per selection mode.\n\n    Used for BOTH arms, so the decoder and RandomDeNovo cannot be scored differently."""\n    truth_ik = query["ik14"]\n    out = dict(molecule_id=query["molecule_id"], ik14=truth_ik, n_cands=len(cands),\n               src_lib=query.get("src_lib", ""), timstof=bool(query.get("timstof", False)),\n               n_ce_known=int(query.get("n_ce_known", 0)))\n    for mode in modes:\n        sel = nd.select_top25(cands, mode, k=k) if cands else []\n        out[f"maxtan_{mode}"] = nd.max_tanimoto(sel, query["can"])\n        keys = [nd.ik14(s) for s, _ in sel]\n        for kk in exact_ks:\n            out[f"exact{kk}_{mode}"] = int(truth_ik in keys[:kk])\n        out[f"nsel_{mode}"] = len(sel)\n    return out\n\n\ndef summarise(scored, label):\n    """-> dict of medians / means / rates over a list of score_molecule records."""\n    df = pd.DataFrame(scored)\n    s = dict(label=label, n=len(df))\n    if not len(df):\n        return s\n    for mode in SELECT_MODES:\n        s[f"median_maxtan_{mode}"] = float(df[f"maxtan_{mode}"].median())\n        s[f"mean_maxtan_{mode}"] = float(df[f"maxtan_{mode}"].mean())\n        for kk in EXACT_KS:\n            s[f"exact{kk}_{mode}"] = float(df[f"exact{kk}_{mode}"].mean())\n    s["median_n_cands"] = float(df["n_cands"].median())\n    s["mean_n_cands"] = float(df["n_cands"].mean())\n    s["zero_cand_share"] = float((df["n_cands"] == 0).mean())\n    return s\n\n\ndef split_summaries(scored, label):\n    """The G3 recorded-not-gating splits: per src_lib, per CE known/unknown, per timsTOF."""\n    df = pd.DataFrame(scored)\n    out = []\n    if not len(df):\n        return out\n    for lib, g in df.groupby("src_lib"):\n        out.append(summarise(g.to_dict("records"), f"{label}|src_lib={lib}"))\n    for known, g in df.groupby(df["n_ce_known"] > 0):\n        out.append(summarise(g.to_dict("records"), f"{label}|ce_known={bool(known)}"))\n    for ts, g in df.groupby("timstof"):\n        out.append(summarise(g.to_dict("records"), f"{label}|timstof={bool(ts)}"))\n    return out\n\n\ndef pick_winner(bake, order=("prefix_seeded", "diverse_beam", "masked_beam")):\n    """The pre-registered selection rule: highest median top-25 max Tanimoto on the dec_val subset\n    under `maxmin`; ties go to A, then C, then B."""\n    key = "median_maxtan_maxmin"\n    best, best_name = None, None\n    for name in order:\n        if name not in bake:\n            continue\n        v = bake[name].get(key)\n        if v is None:\n            continue\n        if best is None or v > best + 1e-12:       # strictly greater, so `order` breaks ties\n            best, best_name = v, name\n    return best_name, best\n\n\nG3_EPS = 1e-9\n\n\ndef g3_verdict(treatment, null, threshold=0.10, converged=None, eps=G3_EPS):\n    """G3.1: median top-25 max Tanimoto beats the null by >= `threshold` on pseudo_c3, under\n    `maxmin`. G3.2: the loss converged (passed in from train_meta.json). Both must hold.\n\n    The comparison carries `eps` because the registered rule is ">= 0.10" on a difference of two\n    binary floats: 0.30 - 0.20 evaluates to 0.09999999999999998, so a delta that is mathematically\n    exactly at the threshold would otherwise FAIL. eps only admits values within 1e-9 of the\n    threshold, far below any difference that could matter to the gate."""\n    t = treatment.get("median_maxtan_maxmin")\n    n = null.get("median_maxtan_maxmin")\n    delta = None if (t is None or n is None) else t - n\n    g31 = bool(delta is not None and delta >= threshold - eps)\n    items = {"G3.1": "PASS" if g31 else "FAIL"}\n    if converged is not None:\n        items["G3.2"] = "PASS" if converged else "FAIL"\n    verdict = "PASS" if all(v == "PASS" for v in items.values()) else "FAIL"\n    return dict(items=items, verdict=verdict, threshold=threshold, eps=eps,\n                treatment_median=t, null_median=n, delta=delta)\n'
NOVEL_P4_SRC = '"""Phase 1 of campaign 20261015-c3-gen-slot2 (c3-gen-union, gate K1): the Class-3 generator union.\n\nTwo generators, one leak-free scale (gate_report 2026-10-15 amendment: no FPNet in the novel channel):\n  * decoder  - `novel_decoder.search_masked_beam` per PREDICTED formula (top-k distinct formulas of the molecule\'s\n               retrieved candidates in the +1.45 ppm-centred window; never the oracle formula);\n  * mutator  - c3v2 analog edits of retrieved anchors (`delta_m_mutator`), scored by `teacher_logp`, the decoder\'s\n               masked teacher-forced log p(SELFIES | spectra, formula). For a beam leaf the two are the same number\n               (test), so the union is ranked on one scale.\nUnion = IK14-deduped (best score kept, sources merged), ranked by score. K1 metrics: top-1 exact rate h, hit@1-4,\nsplit acyclic / ring-bearing and timsTOF.\n\nPure functions only; the Colab driver is build/novel_p4_colab.py.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom collections import Counter\n\nimport numpy as np\n\nimport novel_decoder as nd\n\nFORMULA_K = 3                      # predicted formulas per molecule\nPPM_CENTER, PPM_WIN = 1.45, 10.0   # CLAUDE.md: timsTOF offset; the engines\' own window\n\n\n# --------------------------------------------------------------------------------------------- #\n# Predicted formulas (no oracle): the formulas the retrieval channels already put at the query mass\n# --------------------------------------------------------------------------------------------- #\n\ndef _plain_formula(f):\n    """\'C10H12O3|gen\' / \'C10H12O3\' -> \'C10H12O3\'; anything unparsable -> None."""\n    if not isinstance(f, str):\n        return None\n    f = f.split("|", 1)[0].strip()\n    return f if re.fullmatch(r"(?:[A-Z][a-z]?\\d*)+", f) else None\n\n\ndef predicted_formulas(ranked_formulas, k=FORMULA_K, allowed=None):\n    """Distinct formulas in rank order of the molecule\'s retrieved candidates, first k.\n\n    ranked_formulas: formulas of the fused list in rank order (BASE forms, then ENG / PC formulas).\n    Rank order, not frequency: the first channel hit is the best evidence the pipeline has, and a frequency vote\n    would favour formulas with many isomers in the pool. Formulas with elements outside the decoder\'s vocabulary\n    (`allowed`) are skipped, since the search cannot produce them."""\n    out = []\n    for f in ranked_formulas:\n        p = _plain_formula(f)\n        if p is None or p in out:\n            continue\n        if allowed is not None and not set(nd.parse_formula(p)) <= set(allowed):\n            continue\n        out.append(p)\n        if len(out) >= k:\n            break\n    return out\n\n\n# --------------------------------------------------------------------------------------------- #\n# Teacher-forced masked log p: the decoder\'s score of ANY structure, on the beam\'s own scale\n# --------------------------------------------------------------------------------------------- #\n\ndef teacher_logp(model, memory, mem_pad, table, selfies_str, formula, torch):\n    """Masked cumulative log p of `selfies_str` (+ </s>) under formula-conditioned `memory`.\n\n    Same masks and log-softmax as `search_masked_beam`, so for a beam leaf this equals the leaf\'s logp.\n    -> float, or None when the structure cannot be derived inside `formula` (a token is illegal, or </s> is not\n    legal at the end), i.e. the beam could never have produced it."""\n    vocab = table.vocab\n    try:\n        ids = vocab.encode(selfies_str)\n    except (KeyError, ValueError):\n        return None\n    if not ids:\n        return None\n    counts = nd.parse_formula(formula) if isinstance(formula, str) else dict(formula)\n    t = nd.Tracker(table, {e: int(c) for e, c in counts.items() if c})\n    if t.missing:\n        return None\n    seq = [model.sos_id] + list(ids)\n    tok = torch.tensor([seq], dtype=torch.long, device=memory.device)\n    with torch.no_grad():\n        logits = model.forward(tok, memory, mem_pad)[0].float()       # (L, V): position i predicts seq[i + 1]\n    total = 0.0\n    for i, nxt in enumerate(list(ids) + [model.eos_id]):\n        legal = t.legal_mask()\n        if not legal[nxt]:\n            return None\n        row = logits[i].masked_fill(~torch.from_numpy(legal).to(logits.device), nd.NEG_INF)\n        total += float(torch.log_softmax(row, dim=-1)[nxt])\n        if nxt != model.eos_id:\n            t = t.push(nxt)\n    return total\n\n\ndef to_selfies(smiles):\n    """SMILES -> (SELFIES, formula) with the decoder\'s own training preprocessing (`prep_structure`), or None."""\n    st, _can, s, form = nd.prep_structure(smiles) if isinstance(smiles, str) else ("parse", None, None, None)\n    return (s, form) if st == "ok" else None\n\n\n# --------------------------------------------------------------------------------------------- #\n# Union + K1 metrics\n# --------------------------------------------------------------------------------------------- #\n\ndef union_rank(lists, key_fn):\n    """lists: iterable of (src, [(smiles, score)]). -> [(smiles, ik, score, \'src1+src2\')] best score first.\n\n    IK14-deduped by `key_fn` (the metric\'s tautomer-canonical IK14); a structure found by several generators keeps\n    its best score and every source tag."""\n    best, srcs = {}, {}\n    for src, items in lists:\n        for smi, sc in items:\n            if sc is None or not np.isfinite(sc):\n                continue\n            k = key_fn(smi)\n            if not k:\n                continue\n            srcs.setdefault(k, set()).add(src)\n            if k not in best or sc > best[k][1]:\n                best[k] = (smi, sc)\n    out = [(s, k, sc, "+".join(sorted(srcs[k]))) for k, (s, sc) in best.items()]\n    return sorted(out, key=lambda r: -r[2])\n\n\ndef rank_of(ranked, truth_ik, depth=25):\n    for i, r in enumerate(ranked[:depth], 1):\n        if r[1] == truth_ik:\n            return i\n    return 0\n\n\ndef k1_report(rows):\n    """rows: dicts with rank (0 = miss), acyclic, timstof, has_formula (truth formula among the predicted ones).\n\n    -> h (top-1 exact rate over ALL molecules), hit@1..4, @25, MRR@25, splits, and the formula-recall ceiling."""\n    n = len(rows)\n    r = np.array([x["rank"] for x in rows])\n\n    def block(mask):\n        m = int(mask.sum())\n        rr = r[mask]\n        return dict(n=m, h=float((rr == 1).mean()) if m else float("nan"),\n                    **{f"hit@{k}": float(((rr >= 1) & (rr <= k)).mean()) if m else float("nan") for k in (2, 3, 4, 25)},\n                    mrr25=float(np.where(rr > 0, 1.0 / np.maximum(rr, 1), 0.0).mean()) if m else float("nan"))\n    acyc = np.array([bool(x["acyclic"]) for x in rows])\n    tims = np.array([bool(x["timstof"]) for x in rows])\n    form = np.array([bool(x["has_formula"]) for x in rows])\n    out = dict(all=block(np.ones(n, bool)), acyclic=block(acyc), ring=block(~acyc), timstof=block(tims),\n               formula_recall=float(form.mean()) if n else float("nan"),\n               given_formula=block(form))\n    out["src_of_top1_hits"] = dict(Counter(x.get("src_top1", "") for x in rows if x["rank"] == 1))\n    out["K1"] = "PASS" if out["all"]["h"] >= 0.03 else "DEAD"\n    return out\n'
for fname, text in [("novel_decoder.py", NOVEL_DECODER_SRC), ("simsim_common.py", SIMSIM_COMMON_SRC),
                    ("novel_cohort.py", NOVEL_COHORT_SRC), ("novel_p3_data.py", NOVEL_P3_DATA_SRC),
                    ("novel_p3_eval.py", NOVEL_P3_EVAL_SRC), ("novel_p4.py", NOVEL_P4_SRC)]:
    (SRC / fname).write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC))
import novel_decoder as nd
import novel_p3_data as p3d
import novel_p3_eval as p3e
import novel_p4 as p4
nd.check_constraints()

## Decoder, embeddings, queries, inputs

In [ ]:
CK = torch.load(P3 / "decoder.pt", map_location=DEV, weights_only=False)
VOCAB = nd.SelfiesVocab.from_json(P0 / "selfies_vocab.json")
assert CK["vocab_sha1"] == hashlib.sha1((P0 / "selfies_vocab.json").read_bytes()).hexdigest()
assert tuple(CK["adducts"]) == nd.TEST_ADDUCTS and tuple(CK["ce_bins"]) == nd.CE_BINS
TABLE = nd.TokenTable(VOCAB)
MODEL = nd.DreamsSelfiesDecoder(**CK["hparams"]).to(DEV)
MODEL.load_state_dict(CK["state_dict"])
MODEL.eval()
MAX_LEN = CK["max_len"]
STORE = p3d.EmbeddingStore(EMB, n_rows=N_CORPUS_ROWS)
Q_C3, skipped = p3e.build_queries(pd.read_parquet(P1 / "pseudo_c3.parquet"))
Q = {q["molecule_id"]: q for q in Q_C3}
INP = pickle.load(open(OUT / "p4_inputs.pkl", "rb"))["rows"]
MIDS = sorted(m for m in INP if m in Q)
print(f"decoder epoch {CK['epoch']} | queries {len(Q_C3)} (skipped {len(skipped)}) | inputs {len(INP)} | run {len(MIDS)}")
print("mean predicted formulas", np.mean([len(INP[m]["formulas"]) for m in MIDS]),
      "| mean mutants", np.mean([len(INP[m].get("mutants") or []) for m in MIDS]))


def memory(q, formula):
    return p3e.query_memory(MODEL, dict(q, formula=formula), STORE, DEV, torch)

## Generation (resumable: one pickle per CHUNK molecules under `raw/`)

In [ ]:
def done_ids():
    out = set()
    for f in RAW.glob("chunk_*.pkl"):
        out |= set(pickle.load(open(f, "rb")))
    return out


def run_one(m, beam256):
    q, inp = Q[m], INP[m]
    rec = dict(dec=[], dec256=[], mut=[], secs={})
    t0 = time.time()
    for f in inp["formulas"]:
        mem, pad = memory(q, f)
        rec["dec"] += [(s, lp, f) for s, lp in nd.search_masked_beam(MODEL, mem, f, TABLE, mem_pad=pad, beam=BEAM,
                                                                      max_len=MAX_LEN, alpha=0.0)]
    rec["secs"]["dec"] = time.time() - t0
    if beam256:
        t0 = time.time()
        for f in inp["formulas"]:
            mem, pad = memory(q, f)
            rec["dec256"] += [(s, lp, f) for s, lp in nd.search_masked_beam(MODEL, mem, f, TABLE, mem_pad=pad,
                                                                             beam=256, max_len=MAX_LEN, alpha=0.0)]
        rec["secs"]["dec256"] = time.time() - t0
    t0 = time.time()
    by_f = {}
    for smi, f, ai, dec in inp.get("mutants") or []:
        by_f.setdefault(f, []).append((smi, ai, dec))
    for f, items in by_f.items():
        try:
            mem, pad = memory(q, f)
        except Exception:                                  # a formula outside the decoder's elements
            continue
        for smi, ai, dec in items:
            sel = p4.to_selfies(smi)
            lp = None if sel is None else p4.teacher_logp(MODEL, mem, pad, TABLE, sel[0], f, torch)
            rec["mut"].append((smi, lp, f, ai, dec))
    rec["secs"]["mut"] = time.time() - t0
    return rec


todo = [m for m in MIDS if m not in done_ids()]
sub256 = set(MIDS[:SUB256])
print(f"{len(MIDS) - len(todo)} done, {len(todo)} to go")
buf, t_all = {}, time.time()
t_chunk = t_all
for i, m in enumerate(todo, 1):
    try:
        buf[m] = run_one(m, m in sub256)
    except Exception as e:                                 # one bad molecule must not lose the run
        print(m, "failed:", type(e).__name__, e, flush=True)
        buf[m] = dict(dec=[], dec256=[], mut=[], secs={}, error=repr(e))
    if len(buf) >= CHUNK or i == len(todo):
        pickle.dump(buf, open(RAW / f"chunk_{int(time.time())}_{i}.pkl", "wb"), protocol=4)
        buf = {}
        el = time.time() - t_all
        print(f"{i}/{len(todo)}  {el / i:.1f}s/mol  eta {(len(todo) - i) * el / i / 60:.0f} min", flush=True)
        last, t_chunk = time.time() - t_chunk, time.time()     # the next chunk must fit, with 50% slack
        if i < len(todo) and time.time() - T_START + 1.5 * last > TIME_BUDGET_S:
            print(f"time budget: stopping after {i}; rerun to resume", flush=True)
            break

## Merge for download

In [ ]:
ALL = {}
for f in sorted(RAW.glob("chunk_*.pkl")):
    ALL.update(pickle.load(open(f, "rb")))
pickle.dump(dict(raw=ALL, beam=BEAM, sub256=sorted(sub256), decoder_epoch=CK["epoch"]),
            open(OUT / "p4_raw.pkl", "wb"), protocol=4)
sec = pd.DataFrame([r["secs"] for r in ALL.values() if r.get("secs")])
print(f"p4_raw.pkl: {len(ALL)}/{len(MIDS)} molecules -> download it to E:/casmi/c3_gen/ and run build/novel_p4_score.py")
if len(sec):
    print(f"seconds per molecule ({torch.cuda.get_device_name(0) if DEV.type == 'cuda' else 'cpu'}):",
          sec.describe().loc[["mean", "50%", "max"]].round(2).to_dict())
if set(MIDS) <= set(ALL):
    print("P4_DONE")